# MVP · O tempo de resolução de chamados de serviço público é previsível já na abertura?
**UnB · Engenharia de Produção · Prof. André Luiz Marques Serrano · Daniel de Carvalho Vianna (231013322)**

Este notebook executa **todo** o pipeline: coleta (NYC Open Data · 311 Service Requests, API SODA) → ETL em PySpark (bronze → silver → gold) → gravação no **BigQuery Sandbox** → qualidade (6 dimensões) → análise P1–P5 (incluindo um modelo preditivo com validação temporal) → gráficos PNG → geração automática de `README.md`, `catalogo_dados.md`, `short_paper.tex`, `LICENSE` e um `.zip`.

**Como rodar:** `Ambiente de execução → Executar tudo` (Ctrl+F9). Única interação humana: a janela de autenticação Google da **Célula 2** (logo no início). Depois deixe a aba aberta. A coleta é **retomável**: se o Colab desconectar sem reiniciar o ambiente, basta executar tudo de novo.

Pré-requisito único: ter aberto uma vez https://console.cloud.google.com/bigquery com a conta Google (isso cria o projeto do Sandbox, gratuito e sem cartão).

In [ ]:
# ═══ CÉLULA 1 · Configuração ═══
import os, sys, json, time, glob, shutil, datetime, subprocess, re, math, textwrap, random, csv, io
import urllib.request, urllib.parse, urllib.error

os.environ["TZ"] = "UTC"                            # Python e Spark no mesmo fuso (evita deslocar horários ao coletar timestamps)
if hasattr(time, "tzset"):
    time.tzset()
MODO_TESTE = os.environ.get("MVP_TESTE") == "1"     # uso do autor do notebook: testes com dados sintéticos (não use)
PROJECT_ID = os.environ.get("MVP_PROJECT_ID", "")   # opcional: se vazio, o notebook descobre/cria o projeto sozinho
BQ_DATASET = "mvp_311"
BQ_LOCATION = "US"

WORK = os.environ.get("MVP_WORK") or ("/content/mvp" if os.path.isdir("/content") else os.path.abspath("./mvp_work"))
RAW, WH, PQ, SAIDA = (f"{WORK}/{p}" for p in ("raw", "warehouse", "parquet", "saida"))
for _d in (f"{RAW}/311", WH, PQ, SAIDA):
    os.makedirs(_d, exist_ok=True)

INICIO, FIM = datetime.date(2025, 1, 1), datetime.date(2025, 12, 31)   # recorte por data de abertura (created_date)
BASE_URL = os.environ.get("MVP_BASE_URL", "https://data.cityofnewyork.us")
DATASET_ID = "erm2-nwe9"                                              # 311 Service Requests from 2010 to Present
URL_CSV = f"{BASE_URL}/resource/{DATASET_ID}.csv"
URL_JSON = f"{BASE_URL}/resource/{DATASET_ID}.json"
URL_COLS = f"{BASE_URL}/api/views/{DATASET_ID}/columns.json"
LICENCA_URL = "https://www.nyc.gov/html/data/terms.html"
PAGINA = int(os.environ.get("MVP_PAGINA", 50000))   # máximo por requisição na API SODA 2.1: 50.000
PAUSA_S = float(os.environ.get("MVP_PAUSA", 1.0))     # cortesia entre requisições (sem token de aplicativo)
UA = {"User-Agent": "mvp-unb-311/1.0 (trabalho academico UnB)"}
NOTEBOOK = "MVP_311_PRAZO.ipynb"

COLS_OBRIG = ["unique_key", "created_date", "closed_date", "agency", "complaint_type", "borough", "status"]
COLS_OPC = ["agency_name", "descriptor", "location_type", "incident_zip", "open_data_channel_type"]

# Parâmetros do modelo e do critério de veredito (definidos ANTES de ver os resultados)
ATRASO_H = 168               # "atraso" = resolução em mais de 7 dias (168 h)
CORTE_TREINO = datetime.date(2025, 9, 30)   # treino: aberturas até 30/09/2025; teste: a partir de 01/10/2025
LIM_AUC = 0.80               # C1: ROC-AUC no teste temporal >= 0,80 (atraso previsível na abertura)
LIM_RHO = 0.30               # C2: |correlação de Spearman| mediana entre carga (backlog) e tempo de resolução < 0,30
SEMENTE = 42
print("Modo:", "TESTE (dados sintéticos)" if MODO_TESTE else "PRODUÇÃO", "| pasta de trabalho:", WORK)

## Célula 2 · Autenticação Google e projeto do BigQuery (única interação humana)

In [ ]:
# ═══ CÉLULA 2 · Autenticação + projeto + dataset no BigQuery ═══
BQ = {}
if MODO_TESTE:
    print("MODO_TESTE: autenticação e BigQuery ignorados.")
else:
    from google.colab import auth
    auth.authenticate_user()                      # <- única ação humana
    import google.auth
    from google.cloud import bigquery

    creds, proj_default = google.auth.default()

    def _gcloud(args):
        return subprocess.run(["gcloud"] + args, capture_output=True, text=True)

    proj = PROJECT_ID or proj_default
    if not proj:
        r = _gcloud(["projects", "list", "--filter=lifecycleState:ACTIVE", "--format=value(projectId)"])
        ids = [l.strip() for l in r.stdout.splitlines() if l.strip()]
        if ids:
            proj = ids[0]
            print("Projetos encontrados:", ids, "-> usando", proj, "(para outro, defina PROJECT_ID na Célula 1)")
        else:
            novo = "mvp-311-unb-" + "".join(random.choices("0123456789abcdef", k=6))
            r = _gcloud(["projects", "create", novo])
            if r.returncode == 0:
                proj = novo
                print("Projeto criado:", proj)
    if not proj:
        raise RuntimeError(
            "Não encontrei nem consegui criar um projeto Google Cloud.\n"
            "1) Abra https://console.cloud.google.com/bigquery com a MESMA conta do Colab e aceite os termos (modo Sandbox);\n"
            "2) copie o ID do projeto que aparece no topo da página;\n"
            "3) cole-o em PROJECT_ID na Célula 1 e execute tudo de novo.")
    client = bigquery.Client(project=proj, credentials=creds)
    ds = bigquery.Dataset(f"{proj}.{BQ_DATASET}")
    ds.location = BQ_LOCATION
    client.create_dataset(ds, exists_ok=True)
    list(client.query("SELECT 1").result())      # teste de permissão/faturamento
    BQ = {"client": client, "projeto": proj}
    print(f"BigQuery OK · projeto={proj} · dataset={BQ_DATASET} · local={BQ_LOCATION}")

## Célula 3 · Dependências (PySpark local) e Java

In [ ]:
# ═══ CÉLULA 3 · Dependências ═══
if not MODO_TESTE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"])
    if shutil.which("java") is None:
        subprocess.check_call("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless", shell=True)
import pandas as pd
import matplotlib
matplotlib.use("Agg") if MODO_TESTE else None
import matplotlib.pyplot as plt
print(subprocess.run("java -version 2>&1 | head -1", shell=True, capture_output=True, text=True).stdout.strip())

## Célula 4 · Coleta (NYC Open Data · 311 Service Requests, API SODA, um mês por vez)
Páginas de 50.000 linhas em CSV, com *retry* e *backoff* exponencial, validação do cabeçalho, gravação atômica e retomada (páginas já baixadas são puladas). Ao final, auditoria: para cada mês compara-se o total informado pela API (`count(*)`) com o número de linhas efetivamente gravadas.

In [ ]:
# ═══ CÉLULA 4 · Coleta ═══
def get(url, tries=7, timeout=300):
    ultimo = None
    for i in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            ultimo = e
            if e.code in (400, 404):                       # consulta inválida: repetir não adianta
                raise RuntimeError(f"HTTP {e.code} em {url[:200]}: {e.read()[:300]!r}")
        except Exception as e:
            ultimo = e
        espera = min(120, 2 ** i * 2) + random.random()
        print(f"  tentativa {i + 1} falhou ({ultimo}); aguardando {espera:.0f}s")
        time.sleep(espera)
    raise RuntimeError(f"falha ao baixar {url[:200]}: {ultimo}")

def mk_url(base, **p):
    return base + "?" + urllib.parse.urlencode(p, quote_via=urllib.parse.quote, safe=",:()*")

def norm(nome):
    return re.sub(r"[^a-z0-9]+", "_", nome.strip().lower()).strip("_")

def linhas_csv(caminho):
    with open(caminho, newline="", encoding="utf-8") as f:
        return sum(1 for _ in csv.reader(f)) - 1

def meses(ini, fim):
    a, m = ini.year, ini.month
    while (a, m) <= (fim.year, fim.month):
        prox = (a + (m == 12), 1 if m == 12 else m + 1)
        yield f"{a:04d}-{m:02d}-01", f"{prox[0]:04d}-{prox[1]:02d}-01"
        a, m = prox

T0_COLETA = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

# colunas realmente existentes no dataset (as opcionais ausentes são descartadas; as obrigatórias, não)
try:
    existentes = {c["fieldName"] for c in json.loads(get(URL_COLS, tries=4))}
except Exception as e:
    print("Aviso: não consegui listar as colunas do dataset (", e, "); usando a lista padrão.")
    existentes = None
faltam_obrig = [c for c in COLS_OBRIG if existentes is not None and c not in existentes]
if faltam_obrig:
    raise RuntimeError(f"colunas obrigatórias ausentes no dataset: {faltam_obrig}")
COLS = COLS_OBRIG + [c for c in COLS_OPC if existentes is None or c in existentes]
print("Colunas coletadas:", COLS)
SEL = ",".join(COLS)

def contar_mes(a, b):
    where = f"created_date >= '{a}T00:00:00' AND created_date < '{b}T00:00:00'"
    corpo = get(mk_url(URL_JSON, **{"$select": "count(*)", "$where": where}))
    return int(list(json.loads(corpo)[0].values())[0]), where

def baixar_mes(a, b, where):
    k, off, total_linhas, arqs = 0, 0, 0, []
    while True:
        arq = f"{RAW}/311/{a[:7]}_p{k:04d}.csv"
        if not (os.path.exists(arq) and os.path.getsize(arq) > 0):
            corpo = get(mk_url(URL_CSV, **{"$select": SEL, "$where": where, "$order": "unique_key",
                                           "$limit": PAGINA, "$offset": off}))
            if corpo[:1] in (b"<", b"{", b"["):
                raise RuntimeError(f"resposta inesperada (não é CSV) em {a} página {k}: {corpo[:200]!r}")
            cab = [norm(x) for x in corpo.split(b"\n", 1)[0].decode("utf-8-sig").strip().split(",")]
            if not set(COLS_OBRIG) <= set(cab):
                raise RuntimeError(f"cabeçalho inesperado em {a} página {k}: {cab}")
            with open(arq + ".part", "wb") as f:
                f.write(corpo)
            os.replace(arq + ".part", arq)               # gravação atômica
            time.sleep(PAUSA_S)
        n = linhas_csv(arq)
        arqs.append(arq); total_linhas += n
        if n < PAGINA:
            break
        k += 1; off += PAGINA
    return arqs, total_linhas

AUD = []
for a, b in meses(INICIO, FIM):
    total, where = contar_mes(a, b)
    arqs, n = baixar_mes(a, b, where)
    AUD.append({"mes": a[:7], "total_api": total, "baixadas": n, "arquivos": len(arqs)})
    print(f"{a[:7]}: API={total:,} baixadas={n:,} arquivos={len(arqs)}")
    time.sleep(PAUSA_S)

incompletos = [x["mes"] for x in AUD if x["baixadas"] < 0.999 * x["total_api"]]
arqs_todos = sorted(glob.glob(f"{RAW}/311/*.csv"))
META = {
    "coleta_inicio_utc": T0_COLETA.strftime("%Y-%m-%d %H:%M:%S"),
    "coleta_fim_utc": datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S"),
    "recorte_abertura": f"{INICIO} a {FIM}",
    "n_meses": len(AUD), "linhas_por_pagina": PAGINA,
    "n_arquivos": len(arqs_todos), "mb": round(sum(os.path.getsize(x) for x in arqs_todos) / 1e6, 1),
    "linhas_baixadas": sum(x["baixadas"] for x in AUD), "total_anunciado_api": sum(x["total_api"] for x in AUD),
    "meses_incompletos": incompletos,
    "auditoria_por_mes": AUD,
    "colunas": COLS, "fonte": f"NYC Open Data, dataset {DATASET_ID} (311 Service Requests from 2010 to Present)",
    "formato": "CSV (API SODA 2.1, páginas de 50.000 linhas)", "token_api": "não utilizado",
}
with open(f"{SAIDA}/coleta_meta.json", "w", encoding="utf-8") as f:
    json.dump(META, f, ensure_ascii=False, indent=2)
if incompletos or not arqs_todos:
    raise RuntimeError(f"coleta incompleta: meses {incompletos}. Execute a célula de novo (é retomável).")
print(f"\nColeta OK: {META['linhas_baixadas']:,} linhas em {META['n_arquivos']} arquivos ({META['mb']} MB). "
      f"Total anunciado pela API: {META['total_anunciado_api']:,}.")

## Célula 5 · Spark local + camada bronze (dados como chegaram)

In [ ]:
# ═══ CÉLULA 5 · Spark e bronze ═══
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.master("local[*]").appName("mvp_311")
         .config("spark.driver.memory", os.environ.get("MVP_MEM", "7g"))
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.ansi.enabled", "true")          # mesmo comportamento do ambiente em que o ETL foi desenvolvido
         .config("spark.sql.caseSensitive", "true")
         .config("spark.sql.session.timeZone", "UTC")       # horários da fonte são "relógio de parede" de NYC; só diferenças importam
         .config("spark.sql.warehouse.dir", WH)
         .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
S = "mvp_311"
spark.sql(f"DROP DATABASE IF EXISTS {S} CASCADE")
shutil.rmtree(f"{WH}/{S}.db", ignore_errors=True)   # sobra de execução anterior sem metastore
spark.sql(f"CREATE DATABASE {S}")

def sql(q):                                   # usa @S no lugar do schema (evita problemas de chaves em f-strings)
    return spark.sql(q.replace("@S", S))

def to_pd(df):
    return pd.DataFrame([tuple(r) for r in df.collect()], columns=df.columns)

bronze = (spark.read.option("header", "true").option("multiLine", "true").option("quote", '"').option("escape", '"')
          .csv(f"{RAW}/311/*.csv"))
bronze = bronze.toDF(*[norm(c) for c in bronze.columns])
for c in COLS_OPC:                            # colunas opcionais ausentes viram nulas (o resto do pipeline não quebra)
    if c not in bronze.columns:
        bronze = bronze.withColumn(c, F.lit(None).cast("string"))
bronze.write.mode("overwrite").saveAsTable(f"{S}.bronze_311")
N_BRONZE = spark.table(f"{S}.bronze_311").count()
print(f"bronze_311: {N_BRONZE:,} linhas | colunas: {spark.table(f'{S}.bronze_311').columns}")

## Célula 6 · Camada silver (limpeza, tipagem e regras de negócio)
Regras: (1) `created_date`/`closed_date` convertidas com `TRY_CAST` (valor inválido vira nulo, sem quebrar); (2) textos com *trim* e espaços repetidos colapsados, vazio vira nulo; agência e bairro em maiúsculas; CEP válido só com 5 dígitos; (3) deduplicação por `unique_key` (mantém a linha com `closed_date` mais recente); (4) recorte: abertura entre 01/01/2025 e 31/12/2025; (5) `horas_resolucao = closed − created` em horas (nula se não fechado; negativa é mantida na silver e tratada na análise).

In [ ]:
# ═══ CÉLULA 6 · Silver ═══
def nz(c):
    return F.when(F.length(F.trim(F.col(c))) == 0, F.lit(None)).otherwise(F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))

def tcast(c):
    return F.expr(f"try_cast(trim({c}) as timestamp)")

b = spark.table(f"{S}.bronze_311")
s = b.select(
    nz("unique_key").alias("unique_key"),
    tcast("created_date").alias("created_ts"),
    tcast("closed_date").alias("closed_ts"),
    F.upper(nz("agency")).alias("agency"),
    nz("agency_name").alias("agency_name"),
    nz("complaint_type").alias("complaint_type"),
    nz("descriptor").alias("descriptor"),
    nz("location_type").alias("location_type"),
    F.when(F.trim(F.col("incident_zip")).rlike("^[0-9]{5}$"), F.trim(F.col("incident_zip"))).alias("incident_zip"),
    F.upper(nz("borough")).alias("borough"),
    nz("status").alias("status"),
    nz("open_data_channel_type").alias("canal"),
)
s = s.where(F.col("unique_key").isNotNull())
w = Window.partitionBy("unique_key").orderBy(F.col("closed_ts").desc_nulls_last())
n_antes = s.count()
s = s.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")
s = s.where(F.col("created_ts").isNull() | ((F.to_date("created_ts") >= F.lit(INICIO)) & (F.to_date("created_ts") <= F.lit(FIM))))
s = (s.withColumn("horas_resolucao", (F.col("closed_ts").cast("long") - F.col("created_ts").cast("long")) / F.lit(3600.0))
      .withColumn("flag_fechado", F.col("closed_ts").isNotNull()))
s.write.mode("overwrite").saveAsTable(f"{S}.silver_chamado")
N_SILVER = spark.table(f"{S}.silver_chamado").count()
print(f"silver_chamado: {N_SILVER:,} linhas (bronze com unique_key: {n_antes:,}; duplicatas/fora do recorte removidas: {n_antes - N_SILVER:,})")

## Célula 7 · Camada gold (modelo estrela + carga de trabalho por agência e dia)
`fato_chamado` (grão: um chamado) · `dim_data` · `dim_agencia` · `dim_tipo` · `dim_local` · `agg_agencia_dia`. Na `agg_agencia_dia`, **fila_inicio_dia** é o número de chamados da agência já abertos e ainda não fechados no início do dia (só entram chamados de 2025 que foram fechados depois; por isso é a fila "interna ao recorte" e começa subestimada em janeiro) e **criados_7d_ant** é a soma dos chamados criados nos 7 dias anteriores (informação conhecida na abertura).

In [ ]:
# ═══ CÉLULA 7 · Gold ═══
sil = spark.table(f"{S}.silver_chamado")
NOMES_DIA = ["domingo", "segunda-feira", "terça-feira", "quarta-feira", "quinta-feira", "sexta-feira", "sábado"]

datas = (spark.sql(f"SELECT explode(sequence(to_date('{INICIO}'), to_date('{FIM}'), interval 1 day)) AS data")
         .select(F.date_format("data", "yyyyMMdd").cast("int").alias("sk_data"), "data",
                 F.year("data").alias("ano"), F.quarter("data").alias("trimestre"), F.month("data").alias("mes"),
                 F.dayofmonth("data").alias("dia"), F.dayofweek("data").alias("dia_semana_num"),
                 F.element_at(F.array(*[F.lit(x) for x in NOMES_DIA]), F.dayofweek("data")).alias("dia_semana"),
                 F.dayofweek("data").isin(1, 7).alias("fim_de_semana")))
datas.write.mode("overwrite").saveAsTable(f"{S}.dim_data")

sil.groupBy("agency").agg(F.first("agency_name", ignorenulls=True).alias("agency_name"), F.count("*").alias("n_chamados")) \
   .where(F.col("agency").isNotNull()).write.mode("overwrite").saveAsTable(f"{S}.dim_agencia")
sil.groupBy("complaint_type").agg(F.countDistinct("descriptor").alias("n_descritores"), F.count("*").alias("n_chamados")) \
   .where(F.col("complaint_type").isNotNull()).write.mode("overwrite").saveAsTable(f"{S}.dim_tipo")
sil.groupBy("borough").agg(F.countDistinct("incident_zip").alias("n_cep"), F.count("*").alias("n_chamados")) \
   .where(F.col("borough").isNotNull()).write.mode("overwrite").saveAsTable(f"{S}.dim_local")

# carga de trabalho por agência e dia
d = spark.table(f"{S}.dim_data").select("sk_data", F.col("data").alias("dia"))
ag = spark.table(f"{S}.dim_agencia").select("agency")
ok = sil.where(F.col("created_ts").isNotNull() & F.col("agency").isNotNull())
c_all = ok.groupBy("agency", F.to_date("created_ts").alias("dia")).agg(F.count("*").alias("criados"),
            F.sum(F.col("flag_fechado").cast("int")).alias("criados_fechados"))
c_fech = (ok.where("flag_fechado")
          .groupBy("agency", F.to_date(F.greatest("closed_ts", "created_ts")).alias("dia")).agg(F.count("*").alias("fechados")))
g = (ag.crossJoin(d).join(c_all, ["agency", "dia"], "left").join(c_fech, ["agency", "dia"], "left")
     .select("agency", "sk_data", "dia", *[F.coalesce(F.col(c), F.lit(0)).cast("long").alias(c) for c in ("criados", "criados_fechados", "fechados")]))
wa = Window.partitionBy("agency").orderBy("dia")
g = (g.withColumn("fila_inicio_dia", F.coalesce(F.sum(F.col("criados_fechados") - F.col("fechados")).over(wa.rowsBetween(Window.unboundedPreceding, -1)), F.lit(0)))
      .withColumn("criados_7d_ant", F.coalesce(F.sum("criados").over(wa.rowsBetween(-7, -1)), F.lit(0))))
g.drop("dia").write.mode("overwrite").saveAsTable(f"{S}.agg_agencia_dia")

agg = spark.table(f"{S}.agg_agencia_dia").select("agency", "sk_data", "fila_inicio_dia", "criados_7d_ant")
fato = (sil.where(F.col("created_ts").isNotNull())
        .withColumn("sk_data_abertura", F.date_format("created_ts", "yyyyMMdd").cast("int"))
        .withColumn("hora_abertura", F.hour("created_ts"))
        .join(agg, (sil.agency == agg.agency) & (F.date_format(sil.created_ts, "yyyyMMdd").cast("int") == agg.sk_data), "left")
        .select("unique_key", "sk_data_abertura", "hora_abertura", sil.agency, "complaint_type", "descriptor", "location_type", "borough",
                "incident_zip", "canal", "status", "created_ts", "closed_ts", "horas_resolucao", "flag_fechado",
                F.when(F.col("horas_resolucao") >= 0, F.col("horas_resolucao") > F.lit(float(ATRASO_H))).alias("flag_atraso"),
                F.col("fila_inicio_dia").alias("fila_agencia_inicio_dia"), F.col("criados_7d_ant").alias("criados_agencia_7d_ant")))
fato.write.mode("overwrite").saveAsTable(f"{S}.fato_chamado")

TABELAS = [("bronze_311", "bronze"), ("silver_chamado", "silver"), ("fato_chamado", "gold"), ("dim_data", "gold"),
           ("dim_agencia", "gold"), ("dim_tipo", "gold"), ("dim_local", "gold"), ("agg_agencia_dia", "gold")]
for t, cam in TABELAS:
    print(f"{cam:7s} {t:18s} {spark.table(f'{S}.{t}').count():>10,}")

## Célula 8 · Persistência na nuvem (BigQuery) e prova de gravação
Silver e gold são exportadas do Spark em Parquet e carregadas no BigQuery (*load job*, permitido no Sandbox). Em seguida confere-se, com `SELECT COUNT(*)` no próprio BigQuery, se a contagem de linhas bate com a do Spark. Se não bater, o notebook para.

In [ ]:
# ═══ CÉLULA 8 · BigQuery ═══
TABELAS_BQ = ["silver_chamado", "fato_chamado", "dim_data", "dim_agencia", "dim_tipo", "dim_local", "agg_agencia_dia"]
CAMADA = dict((t, c) for t, c in TABELAS)
PERSIST = []
for t in TABELAS_BQ:
    df = spark.table(f"{S}.{t}")
    n_spark = df.count()
    pdir = f"{PQ}/{t}"
    shutil.rmtree(pdir, ignore_errors=True)
    df.coalesce(1).write.mode("overwrite").parquet(pdir)
    parte = glob.glob(f"{pdir}/part-*.parquet")[0]
    if MODO_TESTE:
        PERSIST.append({"tabela": t, "camada": CAMADA[t], "linhas_spark": n_spark, "linhas_nuvem": n_spark,
                        "mb": round(os.path.getsize(parte) / 1e6, 2)})
        continue
    client, proj = BQ["client"], BQ["projeto"]
    tid = f"{proj}.{BQ_DATASET}.{t}"
    cfg = bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.PARQUET,
                                 write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)
    with open(parte, "rb") as f:
        client.load_table_from_file(f, tid, job_config=cfg).result()
    n_bq = list(client.query(f"SELECT COUNT(*) AS n FROM `{tid}`").result())[0]["n"]
    tab = client.get_table(tid)
    if n_bq != n_spark:
        raise RuntimeError(f"Contagem divergente em {t}: Spark={n_spark} BigQuery={n_bq}")
    PERSIST.append({"tabela": t, "camada": CAMADA[t], "linhas_spark": n_spark, "linhas_nuvem": int(n_bq),
                    "mb": round((tab.num_bytes or 0) / 1e6, 2)})
    print(f"gravada {tid}: {n_bq:,} linhas")

if MODO_TESTE:
    BQ_URL, BQ_LOCAL = "(modo teste: nada foi enviado ao BigQuery)", "ARMAZENAMENTO LOCAL (MODO TESTE)"
else:
    BQ_LOCAL = f"{BQ['projeto']}.{BQ_DATASET}"
    BQ_URL = (f"https://console.cloud.google.com/bigquery?project={BQ['projeto']}"
              f"&p={BQ['projeto']}&d={BQ_DATASET}&page=dataset")
    print("\nTabelas no BigQuery:", [t.table_id for t in BQ["client"].list_tables(f"{BQ['projeto']}.{BQ_DATASET}")])
print("Console do BigQuery:", BQ_URL)

## Célula 9 · Funções auxiliares de formatação e gráficos

In [ ]:
# ═══ CÉLULA 9 · Auxiliares ═══
AZUL, LARANJA, CINZA, VERDE = "#1f5fa8", "#d9730d", "#8a8f98", "#2a9d8f"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})

def fmt(x, d=0):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "n/d"
    return f"{x:,.{d}f}".replace(",", "§").replace(".", ",").replace("§", ".")

def pct(x, d=1):
    return fmt(x, d) + "%"

def fecha(caminho):
    plt.tight_layout()
    plt.savefig(caminho, dpi=150, bbox_inches="tight")
    if not MODO_TESTE:
        plt.show()
    plt.close()

def tabela_png(df, titulo, caminho, larguras, fonte=9, quebra=None):
    quebra = quebra or [40] * len(df.columns)
    cel = [[textwrap.fill(str(v), quebra[j]) for j, v in enumerate(row)] for row in df.values.tolist()]
    nls = [max(x.count("\n") for x in row) + 1 for row in cel]
    alt_pol = 0.22 + 0.2 * (sum(nls) + 1)                       # altura total das linhas, em polegadas
    fig_h = alt_pol + 1.0
    fig, ax = plt.subplots(figsize=(13, fig_h))
    fig.subplots_adjust(top=1 - 0.7 / fig_h, bottom=0.01, left=0.01, right=0.99)
    ax.axis("off")
    tb = ax.table(cellText=cel, colLabels=list(df.columns), loc="upper center", cellLoc="left", colWidths=larguras)
    tb.auto_set_font_size(False); tb.set_fontsize(fonte)
    ax_h = fig_h - 0.7 - 0.01 * fig_h
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor("#c9ccd1")
        nl = 1 if r == 0 else nls[r - 1]
        cell.set_height((0.12 + 0.2 * nl) / ax_h)
        if r == 0:
            cell.set_facecolor(AZUL); cell.set_text_props(color="white", fontweight="bold")
        else:
            cell.set_facecolor("#f4f6f9" if r % 2 == 0 else "white")
    fig.suptitle(titulo, fontsize=12, fontweight="bold", x=0.01, ha="left", y=0.99, va="top")
    fecha(caminho)

def md_tabela(df, alinha=None):
    cab = "| " + " | ".join(df.columns) + " |\n|" + "|".join("---:" if (alinha and alinha[i] == "r") else "---" for i in range(len(df.columns))) + "|\n"
    return cab + "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in df.values.tolist())

## Célula 10 · Qualidade dos dados (6 dimensões) + evidências PNG

In [ ]:
# ═══ CÉLULA 10 · Qualidade ═══
Q, DOM = {}, {}
sc = spark.table(f"{S}.silver_chamado")
n_silver = sc.count()
Q["n_bronze"], Q["n_silver"] = N_BRONZE, n_silver

# 1. Completude
nulos = sc.select([F.round(100 * F.avg(F.col(c).isNull().cast("int")), 2).alias(c) for c in sc.columns]).collect()[0].asDict()
r = sql("""SELECT SUM(CASE WHEN borough = 'UNSPECIFIED' THEN 1 ELSE 0 END) AS bairro_nd,
                  SUM(CASE WHEN NOT flag_fechado THEN 1 ELSE 0 END) AS abertos,
                  SUM(CASE WHEN created_ts IS NOT NULL AND hour(created_ts) = 0 AND minute(created_ts) = 0 AND second(created_ts) = 0 THEN 1 ELSE 0 END) AS cria_0h,
                  SUM(CASE WHEN closed_ts IS NOT NULL AND hour(closed_ts) = 0 AND minute(closed_ts) = 0 AND second(closed_ts) = 0 THEN 1 ELSE 0 END) AS fecha_0h,
                  SUM(CASE WHEN horas_resolucao < 0 THEN 1 ELSE 0 END) AS negativos,
                  SUM(CASE WHEN horas_resolucao = 0 THEN 1 ELSE 0 END) AS zeros,
                  SUM(CASE WHEN horas_resolucao > 8760 THEN 1 ELSE 0 END) AS acima_1ano,
                  MIN(horas_resolucao) AS h_min, MAX(horas_resolucao) AS h_max,
                  MAX(created_ts) AS ult_cria, MAX(closed_ts) AS ult_fecha, MIN(created_ts) AS prim_cria,
                  SUM(CASE WHEN flag_fechado AND status <> 'Closed' THEN 1 ELSE 0 END) AS fechado_status_outro,
                  SUM(CASE WHEN NOT flag_fechado AND status = 'Closed' THEN 1 ELSE 0 END) AS closed_sem_data,
                  SUM(CASE WHEN closed_ts > current_timestamp() THEN 1 ELSE 0 END) AS fecha_futuro,
                  SUM(CASE WHEN created_ts > current_timestamp() THEN 1 ELSE 0 END) AS cria_futuro
           FROM @S.silver_chamado""").collect()[0]
Q.update({k: (r[k] or 0) for k in ("bairro_nd", "abertos", "cria_0h", "fecha_0h", "negativos", "zeros", "acima_1ano",
                                    "fechado_status_outro", "closed_sem_data", "fecha_futuro", "cria_futuro")})
Q.update(h_min=r.h_min, h_max=r.h_max, ult_cria=str(r.ult_cria)[:19], ult_fecha=str(r.ult_fecha)[:19], prim_cria=str(r.prim_cria)[:19])
comp = [(c, float(v), "nulo") for c, v in nulos.items() if c != "flag_fechado"]
comp += [("borough = UNSPECIFIED", 100 * Q["bairro_nd"] / n_silver, "ausência codificada"),
         ("sem data de fechamento (aberto)", 100 * Q["abertos"] / n_silver, "ausência estrutural")]
comp = sorted(comp, key=lambda x: -x[1])
Q["comp"] = comp
Q["pct_aberto"] = 100 * Q["abertos"] / n_silver

fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(comp) + 1.6))
ys = range(len(comp))
ax.barh(list(ys), [c[1] for c in comp], color=[AZUL if c[2] == "nulo" else LARANJA for c in comp])
ax.set_yticks(list(ys)); ax.set_yticklabels([c[0] for c in comp]); ax.invert_yaxis()
mx = max([c[1] for c in comp] + [1])
for y, c in zip(ys, comp):
    ax.text(c[1] + mx * 0.015, y, pct(c[1], 2), va="center", fontsize=9)
ax.set_xlim(0, mx * 1.18); ax.set_xlabel("% de linhas da silver_chamado")
ax.set_title("Qualidade 1/3 · Completude (azul = nulos · laranja = ausência codificada/estrutural)", fontsize=11)
fecha(f"{SAIDA}/qualidade_1_completude.png")

# 2. Unicidade / 3. Conformidade / 4. Consistência
dup_bronze = N_BRONZE - spark.table(f"{S}.bronze_311").select("unique_key").distinct().count()
dup_silver = n_silver - sc.select("unique_key").distinct().count()
fc = spark.table(f"{S}.fato_chamado")
dup_fato = fc.count() - fc.select("unique_key").distinct().count()
dup_dim = sum(spark.table(f"{S}.{t}").count() - spark.table(f"{S}.{t}").select(k).distinct().count()
              for t, k in (("dim_agencia", "agency"), ("dim_tipo", "complaint_type"), ("dim_local", "borough"), ("dim_data", "sk_data")))
Q.update(dup_bronze=dup_bronze, dup_silver=dup_silver, dup_fato=dup_fato, dup_dim=dup_dim)

BAIRROS = ["BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND", "UNSPECIFIED"]
STATUS = ["Closed", "Open", "Pending", "Assigned", "Started", "In Progress", "Unspecified", "Draft"]
CANAIS = ["PHONE", "ONLINE", "MOBILE", "OTHER", "UNKNOWN"]
def fora(col, dom):
    lst = ", ".join("'" + x + "'" for x in dom)
    return sql(f"SELECT COUNT(*) AS n FROM @S.silver_chamado WHERE {col} IS NOT NULL AND {col} NOT IN ({lst})").collect()[0].n
Q["bairro_fora"], Q["status_fora"], Q["canal_fora"] = fora("borough", BAIRROS), fora("status", STATUS), fora("upper(canal)", CANAIS)
Q["id_fora"] = sql("SELECT COUNT(*) AS n FROM @S.silver_chamado WHERE NOT (unique_key RLIKE '^[0-9]+$')").collect()[0].n
Q["cep_invalido"] = spark.table(f"{S}.bronze_311").where(F.col("incident_zip").isNotNull() & ~F.trim(F.col("incident_zip")).rlike("^[0-9]{5}$")).count()
Q["fora_recorte"] = sql(f"SELECT COUNT(*) AS n FROM @S.silver_chamado WHERE created_ts IS NOT NULL AND (to_date(created_ts) < to_date('{INICIO}') OR to_date(created_ts) > to_date('{FIM}'))").collect()[0].n
Q["created_nulo"] = sql("SELECT COUNT(*) AS n FROM @S.silver_chamado WHERE created_ts IS NULL").collect()[0].n
Q["fk_data_orfa"] = sql("SELECT COUNT(*) AS n FROM @S.fato_chamado f LEFT ANTI JOIN @S.dim_data d ON f.sk_data_abertura = d.sk_data").collect()[0].n
Q["fk_ag_orfa"] = sql("SELECT COUNT(*) AS n FROM @S.fato_chamado f LEFT ANTI JOIN @S.dim_agencia d ON f.agency = d.agency WHERE f.agency IS NOT NULL").collect()[0].n
Q["fk_tipo_orfa"] = sql("SELECT COUNT(*) AS n FROM @S.fato_chamado f LEFT ANTI JOIN @S.dim_tipo d ON f.complaint_type = d.complaint_type WHERE f.complaint_type IS NOT NULL").collect()[0].n
Q["fato_vs_silver"] = fc.count() - (n_silver - Q["created_nulo"])
Q["fila_negativa"] = sql("SELECT COUNT(*) AS n FROM @S.agg_agencia_dia WHERE fila_inicio_dia < 0").collect()[0].n

def _s(n, ok="OK", ruim="verificar"):
    return ok if not n else ruim
tab2 = pd.DataFrame([
    ("Unicidade", "unique_key duplicado no bronze (antes da deduplicação)", fmt(Q["dup_bronze"]), _s(Q["dup_bronze"], "OK", "TRATADO: deduplicado na silver")),
    ("Unicidade", "unique_key duplicado na silver / na fato", f"{fmt(Q['dup_silver'])} / {fmt(Q['dup_fato'])}", _s(Q["dup_silver"] + Q["dup_fato"])),
    ("Unicidade", "chaves duplicadas nas dimensões", fmt(Q["dup_dim"]), _s(Q["dup_dim"])),
    ("Conformidade", "unique_key fora do padrão numérico", fmt(Q["id_fora"]), _s(Q["id_fora"])),
    ("Conformidade", "borough fora do domínio (5 bairros + UNSPECIFIED)", fmt(Q["bairro_fora"]), _s(Q["bairro_fora"], "OK", "informativo: ver catálogo")),
    ("Conformidade", "status fora do domínio esperado", fmt(Q["status_fora"]), _s(Q["status_fora"], "OK", "informativo: ver catálogo")),
    ("Conformidade", "canal fora do domínio esperado", fmt(Q["canal_fora"]), _s(Q["canal_fora"], "OK", "informativo: ver catálogo")),
    ("Conformidade", "CEP fora do padrão de 5 dígitos (vira nulo na silver)", fmt(Q["cep_invalido"]), _s(Q["cep_invalido"], "OK", "TRATADO: nulo")),
    ("Conformidade", "abertura fora do recorte / data de abertura inválida", f"{fmt(Q['fora_recorte'])} / {fmt(Q['created_nulo'])}", _s(Q["fora_recorte"] + Q["created_nulo"])),
    ("Conformidade", "chaves órfãs (fato→data, fato→agência, fato→tipo)", f"{fmt(Q['fk_data_orfa'])} / {fmt(Q['fk_ag_orfa'])} / {fmt(Q['fk_tipo_orfa'])}", _s(Q["fk_data_orfa"] + Q["fk_ag_orfa"] + Q["fk_tipo_orfa"])),
    ("Consistência", "fechamento anterior à abertura (horas < 0)", fmt(Q["negativos"]), _s(Q["negativos"], "OK", "TRATADO: excluído das análises de tempo")),
    ("Consistência", "com data de fechamento, mas status diferente de Closed", fmt(Q["fechado_status_outro"]), _s(Q["fechado_status_outro"], "OK", "informativo")),
    ("Consistência", "status Closed sem data de fechamento", fmt(Q["closed_sem_data"]), _s(Q["closed_sem_data"], "OK", "informativo")),
    ("Consistência", "linhas da fato − linhas válidas da silver", fmt(Q["fato_vs_silver"]), _s(Q["fato_vs_silver"])),
    ("Consistência", "fila negativa na agg_agencia_dia", fmt(Q["fila_negativa"]), _s(Q["fila_negativa"])),
], columns=["Dimensão", "Verificação", "Ocorrências", "Situação"])
tabela_png(tab2, "Qualidade 2/3 · Unicidade, conformidade e consistência (silver / gold)",
           f"{SAIDA}/qualidade_2_unicidade_conformidade_consistencia.png", [0.13, 0.5, 0.14, 0.23], quebra=[16, 62, 18, 30])

# 5. Acurácia / 6. Atualidade
tab3 = pd.DataFrame([
    ("Acurácia", "tempo de resolução mínimo / máximo (horas)", f"{fmt(Q['h_min'], 2)} / {fmt(Q['h_max'], 0)}", "informativo"),
    ("Acurácia", "resolução maior que 1 ano (8.760 h)", fmt(Q["acima_1ano"]), "informativo" if Q["acima_1ano"] else "OK"),
    ("Acurácia", "resolução exatamente 0 h (mesmo instante)", fmt(Q["zeros"]), "informativo"),
    ("Acurácia", "aberturas às 00:00:00 (possível data sem hora)", f"{fmt(Q['cria_0h'])} ({pct(100 * Q['cria_0h'] / n_silver, 1)})", "limitação: hora de abertura menos confiável"),
    ("Acurácia", "fechamentos às 00:00:00 (possível data sem hora)", f"{fmt(Q['fecha_0h'])} ({pct(100 * Q['fecha_0h'] / max(1, n_silver - Q['abertos']), 1)} dos fechados)", "limitação: resolução em horas é aproximada"),
    ("Acurácia", "datas de abertura / fechamento no futuro", f"{fmt(Q['cria_futuro'])} / {fmt(Q['fecha_futuro'])}", _s(Q["cria_futuro"] + Q["fecha_futuro"])),
    ("Atualidade", "primeira / última abertura", f"{Q['prim_cria']} / {Q['ult_cria']}", "cobre o recorte"),
    ("Atualidade", "último fechamento registrado", Q["ult_fecha"], "dados até a data da coleta"),
    ("Atualidade", "chamados ainda abertos na coleta", f"{fmt(Q['abertos'])} ({pct(Q['pct_aberto'], 2)})", "censura: excluídos das análises de tempo"),
    ("Atualidade", "período de coleta (UTC)", f"{META['coleta_inicio_utc']} a {META['coleta_fim_utc']}", "informativo"),
], columns=["Dimensão", "Verificação", "Resultado", "Situação"])
tabela_png(tab3, "Qualidade 3/3 · Acurácia e atualidade",
           f"{SAIDA}/qualidade_3_acuracia_atualidade.png", [0.12, 0.42, 0.24, 0.22], quebra=[14, 50, 30, 30])

# domínios observados (para o catálogo)
DOM["borough"] = sorted(x[0] for x in sc.select("borough").distinct().collect() if x[0])
DOM["status"] = sorted(x[0] for x in sc.select("status").distinct().collect() if x[0])
DOM["canal"] = sorted(x[0] for x in sc.select("canal").distinct().collect() if x[0])
DOM["n_agencias"] = spark.table(f"{S}.dim_agencia").count()
DOM["n_tipos"] = spark.table(f"{S}.dim_tipo").count()
DOM["n_cep"] = sc.select("incident_zip").distinct().count()
ex = spark.table(f"{S}.fato_chamado").agg(F.min("hora_abertura"), F.max("hora_abertura"), F.min("fila_agencia_inicio_dia"), F.max("fila_agencia_inicio_dia"),
                                           F.min("criados_agencia_7d_ant"), F.max("criados_agencia_7d_ant")).collect()[0]
DOM["hora"], DOM["fila"], DOM["c7"] = (ex[0], ex[1]), (ex[2], ex[3]), (ex[4], ex[5])
print("Qualidade concluída:", {k: v for k, v in Q.items() if k != "comp"})

## Célula 11 · Análise P1–P4 (perguntas de negócio) e gráficos

In [ ]:
# ═══ CÉLULA 11 · Análise P1–P4 ═══
import numpy as np
from scipy import stats as sps

N_ABERT = sql("SELECT COUNT(*) AS n FROM @S.fato_chamado").collect()[0].n
r0 = sql("""SELECT COUNT(*) AS n, SUM(CASE WHEN flag_atraso THEN 1 ELSE 0 END) AS atr,
                   percentile(horas_resolucao, 0.5) AS med, percentile(horas_resolucao, 0.9) AS p90
            FROM @S.fato_chamado WHERE flag_fechado AND horas_resolucao >= 0""").collect()[0]
N_BASE, N_ATRASO, MED_GERAL, P90_GERAL = r0.n, int(r0.atr or 0), float(r0.med), float(r0.p90)
N_EXCL = N_ABERT - N_BASE
PCT_ATRASO = 100 * N_ATRASO / N_BASE
print(f"chamados abertos no recorte: {N_ABERT:,} | base de tempo (fechados, resolução ≥ 0): {N_BASE:,} | >7 dias: {N_ATRASO:,} ({PCT_ATRASO:.1f}%)")

# ── P1 · volume mensal e fechamento ──
p1 = to_pd(sql("""SELECT d.mes AS mes, COUNT(*) AS chamados, SUM(CASE WHEN f.flag_fechado THEN 1 ELSE 0 END) AS fechados,
                         SUM(CASE WHEN f.flag_atraso THEN 1 ELSE 0 END) AS atrasados,
                         SUM(CASE WHEN f.flag_atraso IS NOT NULL THEN 1 ELSE 0 END) AS validos
                  FROM @S.fato_chamado f JOIN @S.dim_data d ON f.sk_data_abertura = d.sk_data GROUP BY d.mes ORDER BY d.mes"""))
for c in p1.columns:
    p1[c] = p1[c].astype(float)
p1["pct_atraso"] = 100 * p1.atrasados / p1.validos.replace(0, np.nan)
p1["pct_fechado"] = 100 * p1.fechados / p1.chamados
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
x = p1.mes.astype(int)
ax[0].bar(x, p1.fechados, color=AZUL, label="Fechados"); ax[0].bar(x, p1.chamados - p1.fechados, bottom=p1.fechados, color=CINZA, label="Ainda abertos")
for xi, v in zip(x, p1.chamados):
    ax[0].text(xi, v, fmt(v / 1000, 0) + " mil", ha="center", va="bottom", fontsize=8)
ax[0].set_xticks(list(x)); ax[0].set_xticklabels([MESES[i - 1] for i in x]); ax[0].set_ylabel("chamados abertos")
ax[0].set_title("P1 · Chamados abertos por mês (2025)"); ax[0].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2)
ax[1].plot(x, p1.pct_atraso, color=LARANJA, marker="o", lw=2.5)
for xi, v in zip(x, p1.pct_atraso):
    ax[1].annotate(pct(v, 1), (xi, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
ax[1].set_xticks(list(x)); ax[1].set_xticklabels([MESES[i - 1] for i in x]); ax[1].set_ylim(0, max(1, p1.pct_atraso.max() * 1.25))
ax[1].set_ylabel("% resolvidos em mais de 7 dias"); ax[1].set_title("Atrasos (> 7 dias) entre os chamados fechados")
fecha(f"{SAIDA}/p1.png")

# ── P2 · agências e tipos de demanda ──
p2_ag = to_pd(sql("""SELECT agency, COUNT(*) AS chamados, percentile(horas_resolucao, 0.5) AS mediana_h, percentile(horas_resolucao, 0.9) AS p90_h,
                            SUM(CASE WHEN flag_atraso THEN 1 ELSE 0 END) AS atrasados
                     FROM @S.fato_chamado WHERE flag_fechado AND horas_resolucao >= 0 AND agency IS NOT NULL
                     GROUP BY agency ORDER BY chamados DESC"""))
for c in p2_ag.columns[1:]:
    p2_ag[c] = p2_ag[c].astype(float)
p2_ag["pct_total"] = 100 * p2_ag.chamados / N_BASE
p2_ag["pct_atraso"] = 100 * p2_ag.atrasados / p2_ag.chamados
p2_ag["pct_dos_atrasos"] = 100 * p2_ag.atrasados / max(1, N_ATRASO)
p2_tp = to_pd(sql("""SELECT complaint_type AS tipo, COUNT(*) AS chamados, percentile(horas_resolucao, 0.5) AS mediana_h,
                            SUM(CASE WHEN flag_atraso THEN 1 ELSE 0 END) AS atrasados
                     FROM @S.fato_chamado WHERE flag_fechado AND horas_resolucao >= 0 AND complaint_type IS NOT NULL
                     GROUP BY complaint_type ORDER BY atrasados DESC"""))
for c in p2_tp.columns[1:]:
    p2_tp[c] = p2_tp[c].astype(float)
p2_tp["pct_atraso"] = 100 * p2_tp.atrasados / p2_tp.chamados
p2_tp["pct_dos_atrasos"] = 100 * p2_tp.atrasados / max(1, N_ATRASO)
p2_tp["acum"] = p2_tp.pct_dos_atrasos.cumsum()
TOP_AG = p2_ag.head(10).copy()
CONC_AG3 = float(p2_ag.sort_values("atrasados", ascending=False).head(3).pct_dos_atrasos.sum())
CONC_TP10 = float(p2_tp.head(10).pct_dos_atrasos.sum())
MULT = (p2_ag[p2_ag.chamados >= 1000].pct_atraso.max() / max(1e-9, p2_ag[p2_ag.chamados >= 1000].pct_atraso.min())) if (p2_ag.chamados >= 1000).any() else float("nan")
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
t8 = p2_ag.head(8).iloc[::-1]; yy = np.arange(len(t8))
ax[0].barh(yy - 0.2, t8.mediana_h, 0.4, color=AZUL, label="Mediana"); ax[0].barh(yy + 0.2, t8.p90_h, 0.4, color=LARANJA, label="p90")
ax[0].set_yticks(yy); ax[0].set_yticklabels([f"{a} (n={fmt(n)})" for a, n in zip(t8.agency, t8.chamados)], fontsize=8)
ax[0].set_xscale("log"); ax[0].set_xlabel("horas até a resolução (escala log)"); ax[0].legend(frameon=False)
ax[0].set_title("P2 · Tempo de resolução por agência (8 maiores)")
t10 = p2_tp.head(10)
ax[1].bar(range(len(t10)), t10.atrasados, color=AZUL)
ax[1].set_xticks(range(len(t10))); ax[1].set_xticklabels([textwrap.shorten(t, 22, placeholder="…") for t in t10.tipo], rotation=60, ha="right", fontsize=8)
ax[1].set_ylabel("chamados resolvidos em > 7 dias")
a2 = ax[1].twinx(); a2.plot(range(len(t10)), t10.acum, color=LARANJA, marker="o"); a2.set_ylim(0, 100); a2.set_ylabel("% acumulado dos atrasos")
a2.spines["top"].set_visible(False)
ax[1].set_title("Tipos de demanda que mais geram atrasos")
fecha(f"{SAIDA}/p2.png")

# ── P3 · padrões de abertura (dia da semana, hora, mês) ──
def grupo(col, extra=""):
    q = f"""SELECT {col} AS g, COUNT(*) AS n, SUM(CASE WHEN f.flag_atraso THEN 1 ELSE 0 END) AS atr, percentile(f.horas_resolucao, 0.5) AS med
            FROM @S.fato_chamado f JOIN @S.dim_data d ON f.sk_data_abertura = d.sk_data
            WHERE f.flag_fechado AND f.horas_resolucao >= 0 {extra} GROUP BY {col} ORDER BY {col}"""
    t = to_pd(sql(q))
    for c in t.columns:
        t[c] = t[c].astype(float)
    t["pct_atraso"] = 100 * t.atr / t.n
    return t
p3_dow, p3_mes, p3_hora = grupo("d.dia_semana_num"), grupo("d.mes"), grupo("f.hora_abertura")
DOWN = ["dom", "seg", "ter", "qua", "qui", "sex", "sáb"]
AMP_DOW = float(p3_dow.pct_atraso.max() - p3_dow.pct_atraso.min())
AMP_MES = float(p3_mes.pct_atraso.max() - p3_mes.pct_atraso.min())
AMP_HORA = float(p3_hora[p3_hora.n >= 1000].pct_atraso.max() - p3_hora[p3_hora.n >= 1000].pct_atraso.min()) if (p3_hora.n >= 1000).any() else float("nan")
AMP_AG = float(p2_ag[p2_ag.chamados >= 1000].pct_atraso.max() - p2_ag[p2_ag.chamados >= 1000].pct_atraso.min()) if (p2_ag.chamados >= 1000).any() else float("nan")
fig, ax = plt.subplots(1, 3, figsize=(14, 4.4))
ax[0].bar([DOWN[int(i) - 1] for i in p3_dow.g], p3_dow.pct_atraso, color=AZUL); ax[0].set_title("Dia da semana de abertura"); ax[0].set_ylabel("% resolvidos em > 7 dias")
ax[1].plot([MESES[int(i) - 1] for i in p3_mes.g], p3_mes.pct_atraso, color=LARANJA, marker="o", lw=2.5); ax[1].set_title("Mês de abertura")
ax[2].bar(p3_hora.g.astype(int), p3_hora.n, color=CINZA); ax[2].set_title("Hora de abertura (volume e % de atraso)"); ax[2].set_xlabel("hora do dia"); ax[2].set_ylabel("chamados fechados")
b2 = ax[2].twinx(); hh = p3_hora[p3_hora.n >= 1000]; b2.plot(hh.g.astype(int), hh.pct_atraso, color=LARANJA, marker=".", lw=1.5); b2.spines["top"].set_visible(False)
for a in ax[:2]:
    a.set_ylim(0, max(1, max(p3_dow.pct_atraso.max(), p3_mes.pct_atraso.max()) * 1.3))
fig.suptitle("P3 · Padrões temporais de abertura × atraso", fontweight="bold", y=1.02)
fecha(f"{SAIDA}/p3.png")

# ── P4 · carga de trabalho (fila) × tempo de resolução, por agência ──
d4 = to_pd(sql("""SELECT agency, sk_data_abertura AS sk, COUNT(*) AS n, percentile(horas_resolucao, 0.5) AS med_h,
                         MAX(fila_agencia_inicio_dia) AS fila, MAX(criados_agencia_7d_ant) AS c7
                  FROM @S.fato_chamado WHERE flag_fechado AND horas_resolucao >= 0 AND agency IS NOT NULL AND sk_data_abertura >= 20250201
                  GROUP BY agency, sk_data_abertura HAVING COUNT(*) >= 30"""))
for c in d4.columns[1:]:
    d4[c] = d4[c].astype(float)
rho = []
for ag_, g in d4.groupby("agency"):
    if len(g) >= 60 and g.fila.nunique() > 1 and g.med_h.nunique() > 1:
        r1 = sps.spearmanr(g.fila, g.med_h)[0]
        r2 = sps.spearmanr(g.c7, g.med_h)[0] if g.c7.nunique() > 1 else float("nan")
        rho.append((ag_, len(g), float(r1), float(r2)))
p4 = pd.DataFrame(rho, columns=["agency", "dias", "rho_fila", "rho_c7"])
N_AG_P4 = len(p4)
RHO_FILA_ABS = float(p4.rho_fila.abs().median()) if N_AG_P4 else float("nan")
RHO_C7_ABS = float(p4.rho_c7.abs().median()) if N_AG_P4 else float("nan")
RHO_FILA_MED = float(p4.rho_fila.median()) if N_AG_P4 else float("nan")
PCT_AG_FORTE = float(100 * (p4.rho_fila.abs() >= LIM_RHO).mean()) if N_AG_P4 else float("nan")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
if N_AG_P4:
    ax[0].hist(p4.rho_fila.dropna(), bins=np.linspace(-1, 1, 21), color=AZUL, edgecolor="white")
    ax[0].axvline(LIM_RHO, color=LARANJA, ls="--"); ax[0].axvline(-LIM_RHO, color=LARANJA, ls="--")
    ax[0].set_xlabel("correlação de Spearman (fila no início do dia × mediana do tempo)"); ax[0].set_ylabel("nº de agências")
    ax[0].set_title(f"P4 · Uma correlação por agência (n={N_AG_P4}); linhas = ±{LIM_RHO:.2f}")
    top = p4.sort_values("dias", ascending=False).iloc[0].agency
    g = d4[d4.agency == top]
    ax[1].scatter(g.fila, g.med_h, s=14, color=LARANJA, alpha=.7); ax[1].set_yscale("log")
    ax[1].set_xlabel("chamados em fila no início do dia"); ax[1].set_ylabel("mediana do tempo de resolução (h, log)")
    ax[1].set_title(f"Agência com mais dias válidos: {top}")
fecha(f"{SAIDA}/p4.png")
print(f"P4: {N_AG_P4} agências | |rho| mediano (fila)={RHO_FILA_ABS:.2f} | |rho| mediano (criados 7d)={RHO_C7_ABS:.2f}")

## Célula 12 · P5 · Modelo preditivo: o atraso (> 7 dias) é previsível na abertura?
Só entram variáveis **conhecidas no momento da abertura** (agência, tipo, descritor, local, canal, hora, dia da semana, mês e nº de chamados criados pela agência nos 7 dias anteriores). A fila do dia (`fila_inicio_dia`) usa informação do futuro (quais chamados serão fechados) e por isso **não** entra no modelo. Validação **temporal**: treino com aberturas até 30/09/2025 e teste de 01/10 a 31/12/2025. O limiar de decisão é escolhido num período de validação (setembro) e não no teste. Comparação com três referências: prevalência (AUC 0,5), regra "taxa de atraso por tipo de demanda" e regressão logística.

In [ ]:
# ═══ CÉLULA 12 · Modelo ═══
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_score, recall_score, f1_score, roc_curve

MAX_MODELO = 400_000
CATS = ["agency", "complaint_type", "descriptor", "location_type", "borough", "canal"]
NUMS = ["hora_abertura", "dow", "mes", "criados_agencia_7d_ant"]
dm = (spark.table(f"{S}.fato_chamado").where("flag_fechado AND horas_resolucao >= 0 AND flag_atraso IS NOT NULL")
      .join(spark.table(f"{S}.dim_data").select(F.col("sk_data").alias("_sk"), F.col("dia_semana_num").alias("dow"), "mes"), F.col("sk_data_abertura") == F.col("_sk"))
      .select(*CATS, "hora_abertura", "dow", "mes", "criados_agencia_7d_ant", F.to_date("created_ts").alias("dia"), F.col("flag_atraso").cast("int").alias("y")))
frac = min(1.0, MAX_MODELO / max(1, N_BASE))
if frac < 1:
    dm = dm.sample(False, frac, seed=SEMENTE)
pdm = to_pd(dm)
pdm["dia"] = pd.to_datetime(pdm["dia"])
print(f"base do modelo: {len(pdm):,} chamados ({100 * frac:.0f}% da base de tempo), prevalência de atraso {100 * pdm.y.mean():.1f}%")

corte = pd.Timestamp(CORTE_TREINO); corte_val = corte - pd.Timedelta(days=30)
tr_all, te = pdm[pdm.dia <= corte].copy(), pdm[pdm.dia > corte].copy()
if len(te) < 1000 or te.y.nunique() < 2 or tr_all.y.nunique() < 2:
    raise RuntimeError("treino/teste inadequados (poucos dados ou uma só classe); verifique CORTE_TREINO")
tr_fit, va = tr_all[tr_all.dia <= corte_val], tr_all[tr_all.dia > corte_val]

def codificar(ref, dfs, K=200):
    """categorias frequentes do treino -> códigos inteiros; raras/novas -> OUTROS; nulo -> ND"""
    out = [d[CATS + NUMS].copy() for d in dfs]
    for c in CATS:
        niveis = list(ref[c].dropna().value_counts().index[:K]) + ["OUTROS", "ND"]
        for o in out:
            v = o[c].where(o[c].isna() | o[c].isin(niveis), "OUTROS").fillna("ND")
            o[c] = pd.Categorical(v, categories=niveis).codes
    return out

def novo_hgb():
    return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_leaf_nodes=31, l2_regularization=1.0,
                                          categorical_features=[True] * len(CATS) + [False] * len(NUMS), random_state=SEMENTE)

# limiar escolhido na validação (modelo ajustado só com dados até agosto)
Xf, Xv = codificar(tr_fit, [tr_fit, va])
m_val = novo_hgb().fit(Xf, tr_fit.y)
pv = m_val.predict_proba(Xv)[:, 1]
ths = np.unique(np.quantile(pv, np.linspace(0.05, 0.95, 91)))
f1s = [f1_score(va.y, pv >= t) for t in ths]
LIMIAR = float(ths[int(np.argmax(f1s))])

# modelo final (treino até 30/09) avaliado no teste
Xt, Xe = codificar(tr_all, [tr_all, te])
hgb = novo_hgb().fit(Xt, tr_all.y)
s_hgb = hgb.predict_proba(Xe)[:, 1]

# referência 1: taxa de atraso por tipo de demanda (suavizada), aprendida no treino
prior = tr_all.y.mean()
agg_t = tr_all.groupby("complaint_type").y.agg(["sum", "count"])
taxa = (agg_t["sum"] + 20 * prior) / (agg_t["count"] + 20)
s_regra = te.complaint_type.map(taxa).fillna(prior).values

# referência 2: regressão logística com one-hot
def feats_lr(d):
    z = d[CATS + ["hora_abertura", "dow", "mes"]].astype(str).copy()
    z["c7"] = np.log1p(d["criados_agencia_7d_ant"].astype(float))
    return z
pre = ColumnTransformer([("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), CATS + ["hora_abertura", "dow", "mes"]),
                         ("num", "passthrough", ["c7"])])
lr = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=300, C=1.0))]).fit(feats_lr(tr_all), tr_all.y)
s_lr = lr.predict_proba(feats_lr(te))[:, 1]

def metricas(y, s, thr=None):
    m = {"auc": float(roc_auc_score(y, s)), "ap": float(average_precision_score(y, s)), "brier": float(brier_score_loss(y, np.clip(s, 0, 1)))}
    k = max(1, int(0.10 * len(y))); top = np.argsort(-s)[:k]
    m["prec_top10"] = float(np.asarray(y)[top].mean()); m["rec_top10"] = float(np.asarray(y)[top].sum() / max(1, np.sum(y)))
    if thr is not None:
        p = s >= thr
        m.update(prec=float(precision_score(y, p, zero_division=0)), rec=float(recall_score(y, p, zero_division=0)), f1=float(f1_score(y, p, zero_division=0)))
    return m
MET = {"hgb": metricas(te.y.values, s_hgb, LIMIAR), "lr": metricas(te.y.values, s_lr), "regra": metricas(te.y.values, s_regra)}
PREV_TESTE = float(te.y.mean()); PREV_TREINO = float(tr_all.y.mean())
N_TREINO, N_TESTE = len(tr_all), len(te)

# importância por permutação (queda do AUC ao embaralhar cada variável) em amostra do teste
amostra = np.random.RandomState(SEMENTE).choice(len(te), size=min(40000, len(te)), replace=False)
pi = permutation_importance(hgb, Xe.iloc[amostra], te.y.values[amostra], scoring="roc_auc", n_repeats=3, random_state=SEMENTE)
NOMES_VAR = {"agency": "agência", "complaint_type": "tipo de demanda", "descriptor": "descritor", "location_type": "tipo de local", "borough": "bairro",
             "canal": "canal", "hora_abertura": "hora de abertura", "dow": "dia da semana", "mes": "mês", "criados_agencia_7d_ant": "carga: chamados da agência nos 7 dias anteriores"}
IMP = pd.DataFrame({"var": [NOMES_VAR[c] for c in Xe.columns], "queda_auc": pi.importances_mean}).sort_values("queda_auc", ascending=False).reset_index(drop=True)
CARGA_IMP = float(IMP.loc[IMP["var"].str.startswith("carga"), "queda_auc"].iloc[0])
print({k: {a: round(b, 3) for a, b in v.items()} for k, v in MET.items()}, "| limiar", round(LIMIAR, 3))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for nome, s_, cor in (("Gradient boosting", s_hgb, LARANJA), ("Regressão logística", s_lr, AZUL), ("Regra: taxa por tipo", s_regra, VERDE)):
    fpr, tpr, _ = roc_curve(te.y, s_)
    ax[0].plot(fpr, tpr, color=cor, lw=2, label=f"{nome} (AUC {fmt(roc_auc_score(te.y, s_), 3)})")
ax[0].plot([0, 1], [0, 1], color=CINZA, ls="--", label="Aleatório (AUC 0,500)")
ax[0].set_xlabel("taxa de falsos positivos"); ax[0].set_ylabel("taxa de verdadeiros positivos"); ax[0].legend(frameon=False, loc="lower right", fontsize=9)
ax[0].set_title(f"P5 · Curva ROC no teste temporal (out–dez, n={fmt(N_TESTE)})")
ii = IMP.iloc[::-1]
ax[1].barh(ii["var"], ii.queda_auc, color=[LARANJA if v.startswith("carga") else AZUL for v in ii["var"]])
ax[1].set_xlabel("queda do AUC ao embaralhar a variável (importância por permutação)"); ax[1].set_title("O que mais explica o atraso")
for lab in ax[1].get_yticklabels():
    lab.set_fontsize(8)
fecha(f"{SAIDA}/p5.png")

# ── veredito (critério declarado na Célula 1) ──
C1 = bool(MET["hgb"]["auc"] >= LIM_AUC)
C2 = bool(RHO_FILA_ABS == RHO_FILA_ABS and RHO_FILA_ABS < LIM_RHO)     # nan -> não atendido
VEREDITO = "CONFIRMADA" if (C1 and C2) else ("PARCIALMENTE CONFIRMADA" if (C1 or C2) else "REFUTADA")
print(f"AUC={MET['hgb']['auc']:.3f} (C1 {'atendido' if C1 else 'não atendido'}) | |rho| mediano da fila={RHO_FILA_ABS:.2f} (C2 {'atendido' if C2 else 'não atendido'}) | veredito: {VEREDITO}")

## Célula 13 · `persistencia.png` (prova de gravação na nuvem)

In [ ]:
# ═══ CÉLULA 13 · Evidência de persistência ═══
agora = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
tp = pd.DataFrame([(x["camada"], x["tabela"], fmt(x["linhas_spark"]), fmt(x["linhas_nuvem"]), fmt(x["mb"], 2),
                    "OK" if x["linhas_spark"] == x["linhas_nuvem"] else "DIVERGE") for x in PERSIST],
                  columns=["Camada", "Tabela", "Linhas (Spark)", "Linhas (BigQuery, COUNT(*))", "Tamanho (MB)", "Conferência"])
tabela_png(tp, f"Persistência · {BQ_LOCAL} · {agora}\n{BQ_URL}", f"{SAIDA}/persistencia.png",
           [0.1, 0.22, 0.17, 0.25, 0.14, 0.12], fonte=10, quebra=[12, 24, 16, 28, 14, 12])

## Célula 14 · Textos de resultado e discussão (gerados a partir dos números calculados)

In [ ]:
# ═══ CÉLULA 14a · Textos ═══
MESES_PT = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho", "agosto", "setembro", "outubro", "novembro", "dezembro"]
DIAS_PT = ["domingo", "segunda-feira", "terça-feira", "quarta-feira", "quinta-feira", "sexta-feira", "sábado"]

def dias(h):
    return fmt(h / 24, 1)

def fmtp(x, d=3):
    return fmt(x, d)

def txt_p1():
    a, z = p1.loc[p1.chamados.idxmax()], p1.loc[p1.chamados.idxmin()]
    pa, pz = p1.loc[p1.pct_atraso.idxmax()], p1.loc[p1.pct_atraso.idxmin()]
    return (f"Foram abertos {fmt(N_ABERT)} chamados em {INICIO.year} (média de {fmt(N_ABERT / 365)} por dia). O mês de maior volume foi {MESES_PT[int(a.mes) - 1]} "
            f"({fmt(a.chamados)}) e o de menor volume foi {MESES_PT[int(z.mes) - 1]} ({fmt(z.chamados)}). Na data da coleta, {pct(100 - Q['pct_aberto'], 1)} dos chamados já estavam fechados. "
            f"Entre os {fmt(N_BASE)} chamados fechados com tempo de resolução válido, {pct(PCT_ATRASO)} levaram mais de 7 dias, com variação mensal de {pct(pz.pct_atraso)} "
            f"({MESES_PT[int(pz.mes) - 1]}) a {pct(pa.pct_atraso)} ({MESES_PT[int(pa.mes) - 1]}). O tempo mediano de resolução foi de {fmt(MED_GERAL, 1)} h ({dias(MED_GERAL)} dias) e o p90, de {fmt(P90_GERAL, 0)} h ({dias(P90_GERAL)} dias). "
            f"Foram excluídos das análises de tempo {fmt(N_EXCL)} chamados ({pct(100 * N_EXCL / N_ABERT, 1)}): os ainda abertos e os com fechamento anterior à abertura.")

def txt_p2():
    t3 = p2_ag.head(3)
    ag = "; ".join(f"{r.agency} ({pct(r.pct_total)} dos chamados, mediana de {fmt(r.mediana_h, 1)} h, {pct(r.pct_atraso)} acima de 7 dias)" for r in t3.itertuples())
    tp = "; ".join(f"{r.tipo} ({fmt(r.atrasados)} atrasos, {pct(r.pct_atraso)} dos seus chamados)" for r in p2_tp.head(3).itertuples())
    return (f"As três agências com mais chamados fechados são: {ag}. Entre as agências com pelo menos 1.000 chamados, a taxa de atraso varia de "
            f"{pct(p2_ag[p2_ag.chamados >= 1000].pct_atraso.min())} a {pct(p2_ag[p2_ag.chamados >= 1000].pct_atraso.max())}"
            f"{' (a agência de maior taxa tem ' + fmt(MULT, 1) + ' vezes a taxa da de menor taxa)' if MULT == MULT and p2_ag[p2_ag.chamados >= 1000].pct_atraso.min() >= 1 else ''}. Os atrasos são concentrados: três agências respondem por {pct(CONC_AG3)} dos "
            f"{fmt(N_ATRASO)} chamados resolvidos em mais de 7 dias e os dez tipos de demanda com mais atrasos respondem por {pct(CONC_TP10)}. Os tipos que mais geram atrasos são: {tp}.")

def txt_p3():
    dmax, dmin = p3_dow.loc[p3_dow.pct_atraso.idxmax()], p3_dow.loc[p3_dow.pct_atraso.idxmin()]
    mx = max(AMP_DOW, AMP_MES, AMP_HORA if AMP_HORA == AMP_HORA else 0)
    razao = AMP_AG / mx if mx > 0 else float("nan")
    return (f"A taxa de atraso (> 7 dias) varia {fmt(AMP_DOW, 1)} pontos percentuais entre os dias da semana de abertura (maior em {DIAS_PT[int(dmax.g) - 1]}, {pct(dmax.pct_atraso)}; "
            f"menor em {DIAS_PT[int(dmin.g) - 1]}, {pct(dmin.pct_atraso)}), {fmt(AMP_MES, 1)} p.p. entre os meses e {fmt(AMP_HORA, 1)} p.p. entre as horas do dia (considerando horas com ao menos 1.000 chamados). "
            f"Para comparação, a diferença entre as agências é de {fmt(AMP_AG, 1)} p.p., {fmt(razao, 1)} vezes a maior variação temporal. "
            f"Cautela: {pct(100 * Q['cria_0h'] / n_silver, 1)} das aberturas estão registradas exatamente à meia-noite, o que indica registros sem hora e limita a leitura da hora de abertura.")

def txt_p4():
    if not N_AG_P4:
        return "Não houve agências com dias suficientes (≥ 60 dias com ≥ 30 chamados fechados) para estimar a correlação."
    sinal = "positiva" if RHO_FILA_MED > 0.10 else ("negativa" if RHO_FILA_MED < -0.10 else "praticamente nula")
    return (f"Em {N_AG_P4} agências com ao menos 60 dias válidos (cada dia com no mínimo 30 chamados fechados), a correlação de Spearman entre a fila no início do dia "
            f"e a mediana do tempo de resolução dos chamados abertos naquele dia tem mediana {fmt(RHO_FILA_MED, 2)} (associação {sinal}) e mediana do valor absoluto {fmt(RHO_FILA_ABS, 2)}; "
            f"{pct(PCT_AG_FORTE, 0)} das agências têm |ρ| ≥ {fmt(LIM_RHO, 2)}. Usando os chamados criados nos 7 dias anteriores como medida de carga, a mediana do |ρ| é {fmt(RHO_C7_ABS, 2)}. "
            f"Pela Lei de Little (L = λW), com capacidade fixa uma fila maior tenderia a vir acompanhada de tempos maiores; a magnitude observada indica quanto disso aparece nos dados. "
            f"Cautela: a fila usa só chamados abertos em {INICIO.year} que foram fechados depois e começa subestimada em janeiro (por isso o período de análise começa em fevereiro), e correlação não implica causalidade.")

def txt_p5():
    h, l, g = MET["hgb"], MET["lr"], MET["regra"]
    top = ", ".join(f"{r.var} ({fmt(r.queda_auc, 3)})" for r in IMP.head(3).itertuples())
    return (f"Com validação temporal (treino: {fmt(N_TREINO)} chamados abertos até 30/09; teste: {fmt(N_TESTE)} abertos de 01/10 a 31/12; prevalência de atraso de {pct(100 * PREV_TREINO)} no treino e {pct(100 * PREV_TESTE)} no teste), "
            f"o ROC-AUC no teste foi de {fmt(h['auc'], 3)} para o gradient boosting, {fmt(l['auc'], 3)} para a regressão logística e {fmt(g['auc'], 3)} para a regra simples de taxa de atraso por tipo de demanda (aleatório = 0,500). "
            f"O ganho do modelo sobre a regra por tipo é de {fmt(h['auc'] - g['auc'], 3)} em AUC. No limiar escolhido na validação ({fmt(LIMIAR, 2)}), o modelo atinge precisão de {pct(100 * h['prec'])}, revocação de {pct(100 * h['rec'])} e F1 de {fmt(h['f1'], 2)}. "
            f"No decil de maior risco, {pct(100 * h['prec_top10'])} dos chamados de fato atrasam (contra {pct(100 * PREV_TESTE)} na média), e esse decil captura {pct(100 * h['rec_top10'])} de todos os atrasos. "
            f"As variáveis de maior importância por permutação (queda do AUC) são: {top}. A carga de trabalho (chamados da agência nos 7 dias anteriores) tem queda de AUC de {fmt(CARGA_IMP, 3)}. "
            f"O modelo usa uma amostra de {fmt(len(pdm))} chamados ({pct(100 * frac, 0)} da base de tempo).")

def txt_geral():
    c1 = f"C1: ROC-AUC no teste = {fmt(MET['hgb']['auc'], 3)} ({'atende' if C1 else 'não atende'} ≥ {fmt(LIM_AUC, 2)})"
    c2 = (f"C2: mediana do |ρ| entre fila e tempo = {fmt(RHO_FILA_ABS, 2)} ({'atende' if C2 else 'não atende'} < {fmt(LIM_RHO, 2)})")
    if C1 and C2:
        impl = ("O atraso é razoavelmente previsível já na abertura e a carga diária da agência explica pouco do tempo de resolução. Na prática, isso sustenta informar ao cidadão um prazo esperado "
                "por tipo de demanda e agência e concentrar o acompanhamento nos chamados classificados como de alto risco, em vez de tratar a carga do dia como a principal causa dos atrasos.")
    elif C1:
        impl = ("O atraso é previsível na abertura, mas a carga de trabalho mostra associação relevante com o tempo de resolução: além de informar prazos por tipo e agência, "
                "faz sentido monitorar a fila e dimensionar capacidade.")
    elif C2:
        impl = ("A carga de trabalho explica pouco do tempo, mas as variáveis conhecidas na abertura não bastam para prever o atraso com a qualidade exigida; "
                "provavelmente faltam informações sobre o conteúdo do chamado ou sobre o processo interno de cada agência.")
    else:
        impl = ("Nem a previsibilidade na abertura nem a independência em relação à carga foram confirmadas: o tempo de resolução depende de fatores não observados nestas variáveis "
                "e a fila é relevante, o que pede dados de processo (equipes, prioridades, etapas) para uma resposta melhor.")
    return (f"Critério declarado antes da execução: a hipótese é CONFIRMADA se (C1) o ROC-AUC do modelo no teste temporal for maior ou igual a {fmt(LIM_AUC, 2)} e (C2) a mediana do valor absoluto da correlação "
            f"de Spearman entre a fila da agência e o tempo de resolução for menor que {fmt(LIM_RHO, 2)}; PARCIALMENTE CONFIRMADA se apenas um dos dois critérios for atendido; REFUTADA se nenhum for. "
            f"Resultado: {c1}; {c2}. Veredito: hipótese {VEREDITO}. {impl} "
            f"Ressalvas: os dados são de uma única cidade e de um único ano; o fechamento é um registro administrativo (pode não coincidir com a solução real do problema); "
            f"parte das horas de abertura e fechamento é registrada à meia-noite; o AUC mede a capacidade de ordenar chamados por risco, não a calibração das probabilidades; "
            f"e os limiares ({fmt(LIM_AUC, 2)} e {fmt(LIM_RHO, 2)}) são uma escolha do autor, explicitada para que o resultado seja auditável.")

AV = {"P1": "respondida", "P2": "respondida", "P3": "respondida; ressalva: hora de abertura pouco confiável em parte dos registros",
      "P4": "respondida; ressalva: fila interna ao recorte e correlação não é causalidade",
      "P5": "respondida; ressalva: um ano de dados de uma cidade e uma amostra para o treino do modelo"}

In [ ]:
# ═══ CÉLULA 14b · README.md ═══
INI_TXT, FIM_TXT = INICIO.strftime("%d/%m/%Y"), FIM.strftime("%d/%m/%Y")
agora_txt = datetime.date.today().strftime("%d/%m/%Y")
marca = ("> **DADOS SINTÉTICOS — execução de teste do pipeline. Não usar como resultado.**\n\n" if MODO_TESTE else "")

top_comp = "; ".join(f"{c} {pct(v, 1)}" for c, v, _ in Q["comp"][:4])
QUAL = pd.DataFrame([
    ("Completude", "% de nulos por atributo (silver_chamado) e ausências estruturais", f"maiores ausências: {top_comp}",
     "Sem fechamento → fora das análises de tempo (reportado); demais ausências mantidas e tratadas como \"ND\" no modelo"),
    ("Unicidade", "`unique_key` duplicado (bronze, silver, fato) e chaves das dimensões",
     f"bronze: {fmt(Q['dup_bronze'])}; silver: {fmt(Q['dup_silver'])}; fato: {fmt(Q['dup_fato'])}; dimensões: {fmt(Q['dup_dim'])}", "Deduplicação na silver (mantém a linha com `closed_date` mais recente)"),
    ("Conformidade", "padrão do ID, bairro, status, canal, CEP, recorte e chaves órfãs",
     f"ID fora do padrão: {fmt(Q['id_fora'])}; bairro fora do domínio: {fmt(Q['bairro_fora'])}; status fora do esperado: {fmt(Q['status_fora'])}; canal fora do esperado: {fmt(Q['canal_fora'])}; CEP inválido: {fmt(Q['cep_invalido'])}; órfãs: {fmt(Q['fk_data_orfa'] + Q['fk_ag_orfa'] + Q['fk_tipo_orfa'])}",
     "CEP inválido vira nulo; valores novos de status/canal são mantidos e listados no catálogo"),
    ("Consistência", "fechamento ≥ abertura; status × data de fechamento",
     f"fechamento antes da abertura: {fmt(Q['negativos'])}; com fechamento e status ≠ Closed: {fmt(Q['fechado_status_outro'])}; Closed sem data: {fmt(Q['closed_sem_data'])}",
     "Resolução negativa excluída das análises de tempo (mantida na gold)"),
    ("Acurácia", "tempo de resolução extremo; datas sem hora",
     f"resolução entre {fmt(Q['h_min'], 2)} h e {fmt(Q['h_max'], 0)} h; maior que 1 ano: {fmt(Q['acima_1ano'])}; aberturas às 00:00:00: {pct(100 * Q['cria_0h'] / n_silver, 1)}",
     "Mantidos; hora de abertura tratada como menos confiável (limitação)"),
    ("Atualidade", "última abertura / último fechamento / ainda abertos",
     f"abertura {Q['ult_cria']}; fechamento {Q['ult_fecha']}; abertos na coleta: {fmt(Q['abertos'])} ({pct(Q['pct_aberto'], 2)})", "Abertos excluídos das análises de tempo (censura à direita)"),
], columns=["Dimensão", "Verificação", "Resultado", "Tratamento"])

def tab_md(df):
    return md_tabela(df)
t1 = pd.DataFrame({"Mês": [MESES[int(m) - 1] for m in p1.mes], "Chamados": [fmt(v) for v in p1.chamados], "Fechados": [fmt(v) for v in p1.fechados],
                   "% fechados": [pct(v) for v in p1.pct_fechado], "Fechados em > 7 dias": [fmt(v) for v in p1.atrasados], "% em > 7 dias": [pct(v) for v in p1.pct_atraso]})
t2 = pd.DataFrame({"Agência": TOP_AG.agency, "Chamados fechados": [fmt(v) for v in TOP_AG.chamados], "% do total": [pct(v) for v in TOP_AG.pct_total],
                   "Mediana (h)": [fmt(v, 1) for v in TOP_AG.mediana_h], "p90 (h)": [fmt(v, 0) for v in TOP_AG.p90_h], "% > 7 dias": [pct(v) for v in TOP_AG.pct_atraso]})
t2b = pd.DataFrame({"Tipo de demanda": p2_tp.head(8).tipo, "Chamados": [fmt(v) for v in p2_tp.head(8).chamados], "Atrasos (> 7 dias)": [fmt(v) for v in p2_tp.head(8).atrasados],
                    "% do tipo": [pct(v) for v in p2_tp.head(8).pct_atraso], "% dos atrasos": [pct(v) for v in p2_tp.head(8).pct_dos_atrasos]})
t3 = pd.DataFrame({"Dia de abertura": [DOWN[int(i) - 1] for i in p3_dow.g], "Chamados fechados": [fmt(v) for v in p3_dow.n], "Mediana (h)": [fmt(v, 1) for v in p3_dow.med],
                   "% > 7 dias": [pct(v) for v in p3_dow.pct_atraso]})
t4 = (p4.sort_values("dias", ascending=False).head(8).assign(**{"ρ fila": lambda d: [fmt(v, 2) for v in d.rho_fila], "ρ criados 7d": lambda d: [fmt(v, 2) for v in d.rho_c7],
      "Dias válidos": lambda d: [fmt(v) for v in d.dias]})[["agency", "Dias válidos", "ρ fila", "ρ criados 7d"]].rename(columns={"agency": "Agência"})) if N_AG_P4 else pd.DataFrame({"Agência": []})
t5 = pd.DataFrame({"Modelo": ["Gradient boosting (variáveis da abertura)", "Regressão logística (one-hot)", "Regra: taxa de atraso por tipo de demanda"],
                   "ROC-AUC": [fmt(MET[k]["auc"], 3) for k in ("hgb", "lr", "regra")], "AP (PR-AUC)": [fmt(MET[k]["ap"], 3) for k in ("hgb", "lr", "regra")],
                   "Precisão no decil de maior risco": [pct(100 * MET[k]["prec_top10"]) for k in ("hgb", "lr", "regra")],
                   "Atrasos capturados no decil": [pct(100 * MET[k]["rec_top10"]) for k in ("hgb", "lr", "regra")]})
t5b = pd.DataFrame({"Variável": IMP["var"], "Queda do AUC (permutação)": [fmt(v, 3) for v in IMP.queda_auc]})

README = marca + """# MVP — O tempo de resolução de chamados de serviço público é previsível já na abertura?

**Pipeline de dados em nuvem (Google Colab + PySpark + BigQuery Sandbox) com NYC Open Data · 311 Service Requests**
Universidade de Brasília · Departamento de Engenharia de Produção · Prof. André Luiz Marques Serrano
Aluno: Daniel de Carvalho Vianna (matrícula 231013322) · Trabalho individual · Introdução à Ciência de Dados

---

## Resumo
Este MVP analisa @@N_ABERT@@ chamados abertos em @@ANO@@ no serviço 311 da cidade de Nova York e pergunta se o prazo de atendimento pode ser previsto no momento da abertura. Os dados foram coletados pela API pública, processados em PySpark (arquitetura medalhão), gravados no BigQuery e analisados com estatística descritiva, correlações por agência e um modelo de classificação com validação temporal. Resultado principal: @@RESUMO_RES@@ Veredito pelo critério declarado: hipótese **@@VEREDITO@@**.

## Sumário
1. [Objetivo](#1-objetivo) · 2. [Plataforma e arquitetura](#2-plataforma-e-arquitetura) · 3. [Dados](#3-dados) · 4. [Modelagem](#4-modelagem) · 5. [Carga (ETL)](#5-carga-etl) · 6. [Qualidade dos dados](#6-qualidade-dos-dados) · 7. [Resultados e discussão](#7-resultados-e-discussão) · 8. [Autoavaliação](#8-autoavaliação) · 9. [Como reproduzir](#9-como-reproduzir) · 10. [Estrutura do repositório](#10-estrutura-do-repositório)

```mermaid
flowchart LR
    A[API SODA 311 NYC] --> B[raw: CSV mensal]
    B --> C[bronze: Spark]
    C --> D[silver: limpo e tipado]
    D --> E[gold: modelo estrela]
    D --> F[(BigQuery)]
    E --> F
    E --> G[Análise P1-P4 e modelo P5]
    G --> H[PNG, README, paper]
```

## 1. Objetivo

### 1.1 Problema
Serviços públicos urbanos funcionam como um sistema de filas: chamados chegam, esperam e são atendidos por equipes de capacidade limitada. Na Engenharia de Produção, o **lead time** (tempo entre a abertura e a resolução), a fila e o cumprimento de prazos (SLA) são medidas centrais de desempenho. Se o prazo de cada chamado puder ser previsto na abertura, a organização consegue informar prazos realistas ao cidadão, priorizar o acompanhamento dos casos de risco e dimensionar capacidade. Este MVP testa essa possibilidade com dados reais do serviço 311 de Nova York.

**Recorte:** chamados abertos de @@INI@@ a @@FIM@@. Foi escolhido um ano completo e recente, em que a maioria dos chamados já está fechada na data da coleta (@@DATA_COLETA@@).

### 1.2 Hipótese
> O tempo de resolução é determinado principalmente pela natureza do chamado (agência e tipo de demanda), conhecida na abertura, e pouco pela carga de trabalho do dia. Portanto, os atrasos (resolução em mais de 7 dias) são previsíveis já na abertura.

A hipótese é refutável: se um modelo com as variáveis da abertura não conseguir separar os chamados atrasados dos demais, ou se a fila explicar fortemente o tempo de resolução, ela é rejeitada.

**Critério de veredito (definido antes da execução):** C1 = ROC-AUC do modelo no teste temporal (aberturas de 01/10 a 31/12) ≥ @@LIM_AUC@@; C2 = mediana do valor absoluto da correlação de Spearman entre a fila da agência no início do dia e o tempo de resolução (uma correlação por agência) < @@LIM_RHO@@. Confirmada se C1 e C2; parcialmente confirmada se só um; refutada se nenhum. Os limiares são escolha do autor.

### 1.3 Perguntas de negócio
| # | Pergunta | Decisão que apoia |
|---|---|---|
| P1 | Como evoluiu, mês a mês em @@ANO@@, o volume de chamados abertos e a parcela resolvida em mais de 7 dias? | Dimensionar a carga de trabalho e a capacidade |
| P2 | Quais agências e tipos de demanda concentram os atrasos? | Direcionar esforço de melhoria de processo |
| P3 | O momento da abertura (dia da semana, hora, mês) altera a chance de atraso? | Planejar escalas e turnos |
| P4 | A carga de trabalho da agência (fila no início do dia) está associada ao tempo de resolução? | Testar a relação fila–tempo (Lei de Little) e decidir sobre capacidade |
| P5 | É possível prever, na abertura, se um chamado será resolvido em mais de 7 dias? | Informar prazo esperado ao cidadão e priorizar acompanhamento |

> Se alguma pergunta não for plenamente respondida, ela permanece nesta lista e é discutida na autoavaliação (seção 8).

## 2. Plataforma e arquitetura

- **Plataforma:** Google Colab (execução) + **BigQuery Sandbox** (persistência na nuvem; gratuito, sem cartão). Custo de operação: zero (Colab gratuito, API pública sem chave, BigQuery Sandbox). No Sandbox as tabelas expiram por padrão em 60 dias; as evidências em PNG deste repositório registram o estado da carga.
- **Arquitetura medalhão:** raw (CSV mensais no disco do Colab) → bronze (tabela Spark, dado como veio) → silver (limpo/tipado, regras de negócio) → gold (modelo estrela e agregado de carga). **Silver e gold são gravadas no BigQuery**.
- **Linguagens e bibliotecas:** Python (coleta, PySpark local 3.5.3 com modo ANSI ligado, pandas, scipy, scikit-learn) e SQL (qualidade e análise).
- **Execução:** um único notebook, [`@@NOTEBOOK@@`](@@NOTEBOOK@@), executável com "Executar tudo".

## 3. Dados

| Fonte | Conteúdo | Licença / termos | Formato original |
|---|---|---|---|
| [NYC Open Data · 311 Service Requests from 2010 to Present](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9) (API SODA, dataset `erm2-nwe9`) | Chamados de serviço: datas de abertura/fechamento, agência, tipo, local (bairro e CEP), canal, status | Dados abertos da cidade de Nova York; [termos de uso](@@LICENCA_URL@@) | CSV paginado (50.000 linhas por requisição) |

- **Data da coleta:** @@COLETA_INI@@ a @@COLETA_FIM@@ (UTC); registro completo em `coleta_meta.json`.
- **Volume obtido:** @@LINHAS@@ linhas em @@N_ARQ@@ arquivos CSV (@@MB@@ MB), em @@N_MESES@@ meses, sem token de aplicativo (pausa de @@PAUSA@@ s entre requisições). O total informado pela API (`count(*)`, @@TOTAL_API@@) foi conferido mês a mês com o total gravado.
- **Colunas coletadas:** @@COLUNAS@@. Não foram coletados endereço, coordenadas nem texto livre.
- **Dados pessoais (LGPD):** as colunas coletadas não identificam pessoas (apenas agência, tipo de demanda, bairro, CEP e datas). **Dados corporativos:** nenhum.
- Os dados brutos **não** são versionados no GitHub (volume); o notebook reproduz a obtenção.

## 4. Modelagem

Modelo **estrela** na camada gold (grão da fato: um chamado):
`fato_chamado` · `dim_data` · `dim_agencia` · `dim_tipo` · `dim_local` · `agg_agencia_dia` (carga por agência e dia). Diagrama, **Catálogo de Dados completo (domínios, obrigatoriedade e linhagem)** em [`catalogo_dados.md`](catalogo_dados.md).

## 5. Carga (ETL)

O notebook [`@@NOTEBOOK@@`](@@NOTEBOOK@@) executa, em ordem:

| Células | Função |
|---|---|
| 1–3 | Configuração, autenticação Google (única ação humana), projeto/dataset no BigQuery, dependências |
| 4 | Coleta mensal da API SODA (retry com backoff, retomada, auditoria contra `count(*)`) |
| 5 | Spark local e camada **bronze** |
| 6 | Camada **silver** (limpeza, tipagem, regras de negócio) |
| 7 | Camada **gold** (modelo estrela e carga por agência/dia) |
| 8 | Carga **silver + gold no BigQuery** e conferência de contagens |
| 10 | Qualidade (6 dimensões) |
| 11–13 | Análise P1–P4, modelo P5, gráficos e evidência de persistência |
| 14–16 | Geração automática de README, catálogo, paper (.tex) e .zip |

**Regras de negócio aplicadas na transformação**
1. Datas: `created_date` e `closed_date` convertidas com `TRY_CAST` (inválida vira nula). O horário da fonte é o horário local de Nova York, tratado como "relógio de parede"; só as diferenças entre datas são usadas.
2. Textos: *trim*, espaços repetidos colapsados, vazio vira nulo; agência e bairro em maiúsculas; CEP válido apenas com 5 dígitos.
3. Deduplicação: uma linha por `unique_key`, mantendo a de `closed_date` mais recente.
4. Recorte: abertura entre @@INI@@ e @@FIM@@.
5. `horas_resolucao` = fechamento − abertura, em horas (nula se não fechado; negativa mantida na gold e excluída das análises de tempo).
6. `flag_atraso` = resolução maior que @@ATRASO_H@@ horas (7 dias); só definida para chamados fechados com resolução ≥ 0.
7. Carga por agência e dia (`agg_agencia_dia`): `fila_inicio_dia` = chamados da agência já abertos e ainda não fechados no início do dia (apenas chamados de @@ANO@@ que foram fechados depois; começa subestimada em janeiro); `criados_7d_ant` = chamados criados nos 7 dias anteriores.
8. Modelo: usa só variáveis conhecidas na abertura (a `fila_inicio_dia` não entra, por usar informação do futuro); treino até 30/09 e teste a partir de 01/10.

**Persistência na nuvem:** projeto/dataset `@@BQ_LOCAL@@`; console: @@BQ_URL@@. Evidência: ![Persistência](persistencia.png)

## 6. Qualidade dos dados

Executada na célula 10 do notebook (completude, unicidade, conformidade, consistência, acurácia, atualidade).

@@QUAL@@

Evidências: ![Qualidade 1](qualidade_1_completude.png) ![Qualidade 2](qualidade_2_unicidade_conformidade_consistencia.png) ![Qualidade 3](qualidade_3_acuracia_atualidade.png)

## 7. Resultados e discussão

Base de análise: @@N_ABERT@@ chamados abertos de @@INI@@ a @@FIM@@; para as análises de tempo, @@N_BASE@@ chamados fechados com resolução válida (excluídos @@N_EXCL@@: ainda abertos ou com fechamento anterior à abertura).

### P1 — Volume mensal e atrasos
![P1](p1.png)

@@T1@@

**Resultado e discussão:** @@P1@@

### P2 — Agências e tipos de demanda
![P2](p2.png)

@@T2@@

@@T2B@@

**Resultado e discussão:** @@P2@@

### P3 — Momento da abertura
![P3](p3.png)

@@T3@@

**Resultado e discussão:** @@P3@@

### P4 — Carga de trabalho × tempo de resolução
![P4](p4.png)

@@T4@@

**Resultado e discussão:** @@P4@@

### P5 — Modelo preditivo
![P5](p5.png)

@@T5@@

@@T5B@@

**Resultado e discussão:** @@P5@@

### Discussão geral
@@GERAL@@

## 8. Autoavaliação

1. **Perguntas respondidas / não respondidas e por quê:** P1 — @@AV1@@. P2 — @@AV2@@. P3 — @@AV3@@. P4 — @@AV4@@. P5 — @@AV5@@.
2. **Limitações dos dados:**
   - Uma única cidade e um único ano; os resultados podem não se generalizar a outros contextos.
   - "Fechado" é um registro administrativo: um chamado pode ser fechado sem que o problema tenha sido resolvido.
   - @@LIM_HORA@@ das aberturas estão às 00:00:00, o que indica registros sem hora; a hora de abertura e a resolução em horas são aproximadas.
   - @@LIM_ABERTOS@@ dos chamados seguem abertos na coleta e ficam de fora das análises de tempo (censura à direita), o que pode subestimar os tempos longos.
   - A fila é calculada só com chamados de @@ANO@@ que foram fechados depois; não inclui chamados abertos antes do recorte.
   - O modelo foi treinado numa amostra de até 400 mil chamados e avaliado em um único corte temporal.
   - Os limiares do veredito (C1 e C2) são arbitrários, ainda que declarados.
3. **Decisões técnicas que mudaria:** usar BigQuery SQL diretamente em vez de Spark local, que é desnecessário para este volume; coletar com token de aplicativo para reduzir o risco de limitação; persistir os dados brutos na nuvem (por exemplo, no Cloud Storage), não apenas as tabelas silver/gold; validar o modelo em mais de um corte temporal.
4. **Extensões para uso contínuo:** coleta incremental por data de atualização, orquestração agendada, previsão do tempo (regressão ou análise de sobrevivência) em vez de só a classificação de atraso, inclusão de variáveis de processo (equipes, prioridades) e de clima/feriados, monitoramento de desempenho do modelo ao longo do tempo.

## 9. Como reproduzir
1. Abra https://console.cloud.google.com/bigquery com a conta Google que será usada e aceite o modo Sandbox (cria o projeto gratuito).
2. Abra o notebook [`@@NOTEBOOK@@`](@@NOTEBOOK@@) no [Google Colab](https://colab.research.google.com) (Arquivo → Fazer upload de notebook).
3. Execute **Ambiente de execução → Executar tudo** e autorize o acesso à conta Google quando solicitado (única interação).
4. Aguarde (a coleta de cerca de @@N_MESES@@ meses em páginas de 50 mil linhas leva em torno de uma hora, dependendo da API). Se o Colab desconectar, execute tudo de novo: a coleta é retomável enquanto o ambiente não for reiniciado.
5. Os arquivos gerados ficam em `saida/` e são baixados ao final (`saida.zip` e arquivos soltos).

## 10. Estrutura do repositório
Repositório com estrutura plana (todos os arquivos na raiz):
```
README.md · LICENSE
@@NOTEBOOK@@           (pipeline completo: coleta, ETL, BigQuery, qualidade, análise, modelo, geração de documentos)
catalogo_dados.md    (catálogo de dados + diagrama + linhagem)
coleta_meta.json     (data, volume e formato da coleta)
short_paper.tex      (artigo curto para o Overleaf; usa p1.png … p5.png)
persistencia.png · qualidade_1_*.png · qualidade_2_*.png · qualidade_3_*.png · p1.png … p5.png   (evidências)
```
"""
resumo_res = (f"{pct(PCT_ATRASO)} dos chamados fechados levaram mais de 7 dias; o modelo de gradient boosting, com variáveis conhecidas na abertura, atingiu ROC-AUC de "
              f"{fmt(MET['hgb']['auc'], 3)} no teste temporal (regra simples por tipo: {fmt(MET['regra']['auc'], 3)}), e a mediana do |ρ| entre fila e tempo foi {fmt(RHO_FILA_ABS, 2)}.")
SUBS = {
    "N_ABERT": fmt(N_ABERT), "N_BASE": fmt(N_BASE), "N_EXCL": fmt(N_EXCL), "ANO": str(INICIO.year), "INI": INI_TXT, "FIM": FIM_TXT, "DATA_COLETA": META["coleta_fim_utc"][:10],
    "RESUMO_RES": resumo_res, "VEREDITO": VEREDITO, "LIM_AUC": fmt(LIM_AUC, 2), "LIM_RHO": fmt(LIM_RHO, 2), "NOTEBOOK": NOTEBOOK, "LICENCA_URL": LICENCA_URL,
    "COLETA_INI": META["coleta_inicio_utc"], "COLETA_FIM": META["coleta_fim_utc"], "LINHAS": fmt(META["linhas_baixadas"]), "N_ARQ": fmt(META["n_arquivos"]), "MB": fmt(META["mb"], 1),
    "N_MESES": str(META["n_meses"]), "PAUSA": fmt(PAUSA_S, 1), "TOTAL_API": fmt(META["total_anunciado_api"]), "COLUNAS": ", ".join(f"`{c}`" for c in META["colunas"]),
    "ATRASO_H": str(ATRASO_H), "BQ_LOCAL": BQ_LOCAL, "BQ_URL": BQ_URL, "QUAL": md_tabela(QUAL),
    "T1": md_tabela(t1), "T2": md_tabela(t2), "T2B": md_tabela(t2b), "T3": md_tabela(t3), "T4": md_tabela(t4) if N_AG_P4 else "_(sem agências com dias suficientes)_",
    "T5": md_tabela(t5), "T5B": md_tabela(t5b), "P1": txt_p1(), "P2": txt_p2(), "P3": txt_p3(), "P4": txt_p4(), "P5": txt_p5(), "GERAL": txt_geral(),
    "AV1": AV["P1"], "AV2": AV["P2"], "AV3": AV["P3"], "AV4": AV["P4"], "AV5": AV["P5"],
    "LIM_HORA": pct(100 * Q["cria_0h"] / n_silver, 1), "LIM_ABERTOS": pct(Q["pct_aberto"], 1),
}
for _k, _v in SUBS.items():
    README = README.replace(f"@@{_k}@@", str(_v))
assert "@@" not in README, "token não substituído no README: " + ", ".join(sorted(set(re.findall(r"@@\w+@@", README))))
with open(f"{SAIDA}/README.md", "w", encoding="utf-8") as f:
    f.write(README)
print("README.md:", len(README), "caracteres")

In [ ]:
# ═══ CÉLULA 15a · catalogo_dados.md ═══
def dom_txt(lst, n=12):
    return ", ".join(lst[:n]) + (f" … (+{len(lst) - n})" if len(lst) > n else "")
def pn(c):
    v = nulos.get(c)
    return f"Sim ({pct(v, 2)} na silver)" if v else "Não"
CAT = f"""# Catálogo de Dados

Plataforma: Google Colab (PySpark) → **BigQuery Sandbox** (`{BQ_LOCAL}`). Modelo: **estrela** na camada gold. Recorte: chamados abertos de {INI_TXT} a {FIM_TXT}.
Colunas "observado" foram calculadas pelo notebook sobre os dados reais carregados.

## Diagrama do modelo

```mermaid
erDiagram
    dim_data ||--o{{ fato_chamado : "sk_data = sk_data_abertura"
    dim_agencia ||--o{{ fato_chamado : "agency"
    dim_tipo ||--o{{ fato_chamado : "complaint_type"
    dim_local ||--o{{ fato_chamado : "borough"
    dim_agencia ||--o{{ agg_agencia_dia : "agency"
    dim_data ||--o{{ agg_agencia_dia : "sk_data"
```

## Linhagem geral

| Origem | Como foi obtida | Raw | Bronze | Silver | Gold |
|---|---|---|---|---|---|
| NYC Open Data · 311 Service Requests (`{DATASET_ID}`) | Notebook `{NOTEBOOK}`, célula 4: CSV mensal pela API SODA, 50.000 linhas por página, sem token | `raw/311/*.csv` | `bronze_311` | `silver_chamado` | `fato_chamado`, `dim_agencia`, `dim_tipo`, `dim_local`, `agg_agencia_dia` |
| Calendário | Gerado por SQL (`sequence`) entre {INICIO} e {FIM} | — | — | — | `dim_data` |

Data da coleta: **{META['coleta_inicio_utc']} a {META['coleta_fim_utc']} UTC** · {fmt(META['linhas_baixadas'])} linhas, {fmt(META['mb'], 1)} MB.
Tabelas no BigQuery: {", ".join(f"`{t}`" for t in TABELAS_BQ)}.

## fato_chamado (grão: um chamado) — {fmt(N_ABERT)} linhas

| Atributo | Tipo | Descrição | Domínio esperado | Domínio observado | Nulos? | Linhagem |
|---|---|---|---|---|---|---|
| unique_key | texto | Identificador do chamado | numérico, único | {fmt(N_ABERT)} distintos | Não | `unique_key` |
| sk_data_abertura | inteiro | Chave para `dim_data` (yyyyMMdd) | {INICIO:%Y%m%d} a {FIM:%Y%m%d} | {Q['prim_cria'][:10].replace('-', '')} a {Q['ult_cria'][:10].replace('-', '')} | Não | `created_date` → data |
| hora_abertura | inteiro | Hora do dia da abertura | 0 a 23 | {DOM['hora'][0]} a {DOM['hora'][1]} | Não | `created_date` → hora |
| agency | texto | Sigla da agência responsável | maiúsculas | {DOM['n_agencias']} agências | {pn('agency')} | `agency` |
| complaint_type | texto | Tipo de demanda | texto livre padronizado | {DOM['n_tipos']} tipos | {pn('complaint_type')} | `complaint_type` |
| descriptor | texto | Detalhe do tipo de demanda | texto | — | {pn('descriptor')} | `descriptor` |
| location_type | texto | Tipo de local | texto | — | {pn('location_type')} | `location_type` |
| borough | categórico | Bairro (borough) | BRONX, BROOKLYN, MANHATTAN, QUEENS, STATEN ISLAND, UNSPECIFIED | {dom_txt(DOM['borough'])} | {pn('borough')} | `borough` |
| incident_zip | texto | CEP do incidente | 5 dígitos | {fmt(DOM['n_cep'])} valores distintos | {pn('incident_zip')} (CEP inválido vira nulo) | `incident_zip` |
| canal | categórico | Canal de abertura | PHONE, ONLINE, MOBILE, OTHER, UNKNOWN | {dom_txt(DOM['canal'])} | {pn('canal')} | `open_data_channel_type` |
| status | categórico | Situação do chamado | Closed, Open, Pending, Assigned, Started, In Progress, Unspecified, Draft… | {dom_txt(DOM['status'])} | {pn('status')} | `status` |
| created_ts | timestamp | Data/hora de abertura | dentro do recorte | {Q['prim_cria']} a {Q['ult_cria']} | Não | `created_date` |
| closed_ts | timestamp | Data/hora de fechamento | ≥ abertura | até {Q['ult_fecha']} | Sim: nulo = ainda aberto ({pct(Q['pct_aberto'], 2)}) | `closed_date` |
| horas_resolucao | numérico | Fechamento − abertura, em horas | ≥ 0 | {fmt(Q['h_min'], 2)} a {fmt(Q['h_max'], 0)} ({fmt(Q['negativos'])} negativos) | Sim: nulo = ainda aberto | calculado |
| flag_fechado | booleano | Tem data de fechamento | true/false | {fmt(N_ABERT - Q['abertos'])} verdadeiros | Não | calculado |
| flag_atraso | booleano | Resolução > {ATRASO_H} h (7 dias) | true/false | {fmt(N_ATRASO)} verdadeiros | Sim: nulo se aberto ou resolução negativa | calculado |
| fila_agencia_inicio_dia | inteiro | Chamados da agência abertos e ainda não fechados no início do dia da abertura | ≥ 0 | {fmt(DOM['fila'][0])} a {fmt(DOM['fila'][1])} | Não | `agg_agencia_dia` |
| criados_agencia_7d_ant | inteiro | Chamados criados pela agência nos 7 dias anteriores | ≥ 0 | {fmt(DOM['c7'][0])} a {fmt(DOM['c7'][1])} | Não | `agg_agencia_dia` |

## dim_data — {fmt((FIM - INICIO).days + 1)} linhas

| Atributo | Tipo | Descrição | Domínio |
|---|---|---|---|
| sk_data | inteiro | Chave (yyyyMMdd) | {INICIO:%Y%m%d} – {FIM:%Y%m%d} |
| data | data | Data completa | {INICIO} a {FIM} |
| ano / trimestre / mes / dia | inteiro | Componentes | {INICIO.year} / 1–4 / 1–12 / 1–31 |
| dia_semana_num / dia_semana | inteiro / texto | 1 = domingo … 7 = sábado; nome em português | 1–7 |
| fim_de_semana | booleano | Sábado ou domingo | true/false |

## dim_agencia — {DOM['n_agencias']} linhas
`agency` (chave), `agency_name` (nome da agência, primeiro valor não nulo), `n_chamados`.

## dim_tipo — {DOM['n_tipos']} linhas
`complaint_type` (chave), `n_descritores` (descritores distintos), `n_chamados`.

## dim_local — {len(DOM['borough'])} linhas
`borough` (chave), `n_cep` (CEPs distintos), `n_chamados`.

## agg_agencia_dia (agência × dia)
`agency`, `sk_data`, `criados`, `criados_fechados`, `fechados` (por data efetiva de fechamento), `fila_inicio_dia` (soma, nos dias anteriores, de `criados_fechados − fechados`; só chamados do recorte que foram fechados depois) e `criados_7d_ant` (soma de `criados` nos 7 dias anteriores).

## silver_chamado — {fmt(n_silver)} linhas
Atributos do fato mais as colunas originais tipadas. Nulos observados (%): {", ".join(f"{c} {pct(v, 2)}" for c, v in nulos.items() if c != "flag_fechado")}.
"""
with open(f"{SAIDA}/catalogo_dados.md", "w", encoding="utf-8") as f:
    f.write(CAT)

In [ ]:
# ═══ CÉLULA 15b · short_paper.tex ═══
def tex_esc(t):
    t = str(t)
    for a, b in (("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"), ("#", r"\#"), ("_", r"\_"),
                 ("{", r"\{"), ("}", r"\}"), ("~", r"\textasciitilde{}"), ("^", r"\textasciicircum{}"),
                 ("≥", r"$\geq$"), ("≤", r"$\leq$"), ("→", r"$\rightarrow$"), ("−", "-"), ("ρ", r"$\rho$"), ("λ", r"$\lambda$")):
        t = t.replace(a, b)
    t = re.sub(r"\[\[(\w+)\]\]", lambda m: r"\cite{" + m.group(1) + "}", t)
    return t

def tex_tab(df, cols_align, legenda, rotulo):
    lin = [r"\begin{table}[ht]", r"\centering\small", r"\begin{tabular}{" + cols_align + "}", r"\toprule",
           " & ".join(tex_esc(c) for c in df.columns) + r" \\", r"\midrule"]
    lin += [" & ".join(tex_esc(v) for v in row) + r" \\" for row in df.values.tolist()]
    lin += [r"\bottomrule", r"\end{tabular}", r"\caption{" + tex_esc(legenda) + r"}", r"\label{" + rotulo + "}", r"\end{table}"]
    return "\n".join(lin)

def tex_fig(arq, legenda, rotulo):
    return "\n".join([r"\begin{figure}[ht]", r"\centering", r"\includegraphics[width=0.85\linewidth]{" + arq + "}",
                      r"\caption{" + tex_esc(legenda) + r"}", r"\label{" + rotulo + "}", r"\end{figure}"])

_ta = pd.DataFrame({"Agência": TOP_AG.head(6).agency, "Chamados": [fmt(v) for v in TOP_AG.head(6).chamados], "Mediana (h)": [fmt(v, 1) for v in TOP_AG.head(6).mediana_h],
                    "p90 (h)": [fmt(v, 0) for v in TOP_AG.head(6).p90_h], "% > 7 dias": [pct(v) for v in TOP_AG.head(6).pct_atraso]})
_tm = pd.DataFrame({"Modelo": ["Gradient boosting", "Regressão logística", "Regra por tipo de demanda"],
                    "ROC-AUC": [fmt(MET[k]["auc"], 3) for k in ("hgb", "lr", "regra")], "AP": [fmt(MET[k]["ap"], 3) for k in ("hgb", "lr", "regra")],
                    "Precisão no decil": [pct(100 * MET[k]["prec_top10"]) for k in ("hgb", "lr", "regra")]})
_marca = (r"\textbf{DADOS SINTÉTICOS --- teste do pipeline; não usar como resultado.}" + "\n\n") if MODO_TESTE else ""

TEX = r"""\documentclass[11pt,a4paper]{article}
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage[brazilian]{babel}
\usepackage{graphicx,booktabs,float}
\usepackage[a4paper,margin=2.5cm]{geometry}
\usepackage[hidelinks]{hyperref}
\title{O tempo de resolução de chamados de serviço público é previsível já na abertura? Evidências do serviço 311 de Nova York (@@ANO@@)}
\author{Daniel de Carvalho Vianna\\ Departamento de Engenharia de Produção, Universidade de Brasília (UnB)}
\date{@@DATA@@}
\begin{document}
\maketitle
@@MARCA@@
\begin{abstract}
@@ABSTRACT@@
\end{abstract}

\section{Introdução}
@@INTRO@@

\section{Revisão da literatura}
@@REVISAO@@

\section{Método}
\subsection{Dados e coleta}
@@DADOS@@

\subsection{Pipeline e regras de negócio}
@@PIPE@@

\subsection{Critério de avaliação da hipótese}
@@CRITERIO@@

\section{Resultados}
\subsection{Volume mensal e atrasos (P1)}
@@P1@@
@@FIGP1@@
\subsection{Agências e tipos de demanda (P2)}
@@P2@@
@@TABA@@
@@FIGP2@@
\subsection{Momento da abertura (P3)}
@@P3@@
@@FIGP3@@
\subsection{Carga de trabalho e tempo de resolução (P4)}
@@P4@@
@@FIGP4@@
\subsection{Modelo preditivo (P5)}
@@P5@@
@@TABM@@
@@FIGP5@@

\section{Discussão}
@@GERAL@@

\section{Qualidade dos dados e limitações}
@@QUAL@@
\begin{itemize}
@@LIMITES@@
\end{itemize}

\section{Conclusão}
@@CONCL@@

\begin{thebibliography}{9}
\bibitem{nyc311} City of New York. \textit{311 Service Requests from 2010 to Present}. NYC Open Data, conjunto \texttt{erm2-nwe9}. \url{https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9}.
\bibitem{little} Little, J. D. C. A proof for the queuing formula: $L = \lambda W$. \textit{Operations Research}, 9(3), 383--387, 1961.
\bibitem{hopp} Hopp, W. J.; Spearman, M. L. \textit{Factory Physics}. 3rd ed. Waveland Press, 2008.
\bibitem{friedman} Friedman, J. H. Greedy function approximation: a gradient boosting machine. \textit{Annals of Statistics}, 29(5), 1189--1232, 2001.
\bibitem{sklearn} Pedregosa, F. et al. Scikit-learn: machine learning in Python. \textit{Journal of Machine Learning Research}, 12, 2825--2830, 2011.
\end{thebibliography}
\end{document}
"""
_lim = [
    "Uma única cidade e um único ano; os resultados podem não se generalizar.",
    "O fechamento é um registro administrativo e pode não coincidir com a solução efetiva do problema.",
    f"{pct(100 * Q['cria_0h'] / n_silver, 1)} das aberturas estão registradas à meia-noite (registros sem hora).",
    f"{pct(Q['pct_aberto'], 1)} dos chamados seguiam abertos na coleta e foram excluídos das análises de tempo (censura à direita).",
    "A fila considera apenas chamados de " + str(INICIO.year) + " fechados depois, e o modelo foi avaliado em um único corte temporal.",
    f"Os limiares do critério de veredito ({fmt(LIM_AUC, 2)} para o AUC e {fmt(LIM_RHO, 2)} para a correlação) são uma escolha do autor.",
]
_sub = {
    "@@ANO@@": str(INICIO.year), "@@DATA@@": datetime.date.today().strftime("%d/%m/%Y"), "@@MARCA@@": _marca,
    "@@ABSTRACT@@": tex_esc(f"Este trabalho investiga se o tempo de resolução de chamados de serviço público pode ser previsto no momento da abertura, usando {fmt(N_ABERT)} chamados abertos em {INICIO.year} no serviço 311 de Nova York, obtidos como dados abertos. "
                           f"Os dados foram coletados por API, processados em PySpark e persistidos no BigQuery. {pct(PCT_ATRASO)} dos chamados fechados levaram mais de 7 dias; um modelo de gradient boosting com variáveis conhecidas na abertura atingiu ROC-AUC de {fmt(MET['hgb']['auc'], 3)} em teste temporal, contra {fmt(MET['regra']['auc'], 3)} de uma regra simples por tipo de demanda, e a correlação entre fila e tempo foi de {fmt(RHO_FILA_ABS, 2)} (mediana do valor absoluto, por agência). "
                           f"Segundo o critério declarado, a hipótese é {VEREDITO}."),
    "@@INTRO@@": tex_esc("Serviços públicos urbanos operam como sistemas de filas: chamados chegam, aguardam e são atendidos por equipes de capacidade limitada. Prazo de atendimento, fila e cumprimento de prazos são medidas centrais na gestão de operações. "
                         "Se o prazo de cada chamado puder ser antecipado na abertura, a organização pode informar prazos realistas, priorizar o acompanhamento de casos de risco e dimensionar capacidade. "
                         "Este trabalho testa, com dados reais do serviço 311 de Nova York [[nyc311]], a hipótese de que o tempo de resolução é determinado principalmente pela natureza do chamado, conhecida na abertura, e pouco pela carga de trabalho do dia."),
    "@@REVISAO@@": tex_esc("A Lei de Little [[little]] relaciona o número médio de itens em um sistema (L), a taxa média de chegada (λ) e o tempo médio de permanência (W) por L = λW, e é um resultado básico da análise de filas e de fluxo em operações [[hopp]]. "
                           "Aplicada a um serviço de atendimento, sugere que, com capacidade fixa, filas maiores tendem a vir acompanhadas de tempos de resolução maiores, o que motiva a pergunta sobre a relação entre fila e tempo. "
                           "Para prever o tempo ou o atraso de cada item, métodos de aprendizado supervisionado como o gradient boosting [[friedman]], disponíveis em bibliotecas como a scikit-learn [[sklearn]], são amplamente usados com dados tabulares que misturam variáveis categóricas e numéricas. "
                           "Os registros de chamados 311 da cidade de Nova York [[nyc311]], publicados como dados abertos, permitem examinar essas ideias com dados reais. Este trabalho não propõe um método novo: aplica essa combinação de análise de fluxo e modelo preditivo a um ano de chamados, com validação temporal."),
    "@@DADOS@@": tex_esc(f"Foram coletados, pela API SODA do NYC Open Data [[nyc311]], os chamados abertos entre {INI_TXT} e {FIM_TXT}, em páginas mensais de {fmt(PAGINA)} linhas em CSV, sem token de aplicativo, com retentativas com espera exponencial e retomada. "
                         f"Foram obtidas {fmt(META['linhas_baixadas'])} linhas ({fmt(META['mb'], 1)} MB) entre {META['coleta_inicio_utc']} e {META['coleta_fim_utc']} (UTC), conferidas mês a mês contra o total informado pela API. "
                         "Foram usadas apenas colunas sem identificação de pessoas (agência, tipo de demanda, bairro, CEP, canal, status e datas). Os dados são públicos."),
    "@@PIPE@@": tex_esc("O pipeline foi executado em um notebook do Google Colab, com ETL em PySpark local (arquitetura medalhão: raw, bronze, silver, gold) e persistência das camadas silver e gold no BigQuery Sandbox, com conferência das contagens de linhas. "
                        "O modelo gold é uma estrela com a tabela fato fato_chamado (grão: um chamado) e as tabelas dim_data, dim_agencia, dim_tipo, dim_local e agg_agencia_dia (carga por agência e dia). "
                        f"Regras: datas convertidas com TRY_CAST; deduplicação por unique_key; resolução em horas = fechamento − abertura; atraso = resolução acima de {ATRASO_H} horas (7 dias); resoluções negativas e chamados ainda abertos excluídos das análises de tempo. "
                        "A fila no início do dia usa apenas chamados do recorte que foram fechados depois. O modelo usa somente variáveis conhecidas na abertura (agência, tipo, descritor, local, canal, hora, dia da semana, mês e chamados da agência nos 7 dias anteriores), com treino até 30/09 e teste de 01/10 a 31/12; o limiar de decisão é escolhido em um período de validação, e a importância das variáveis é medida por permutação. "
                        "São comparados o gradient boosting [[friedman]], uma regressão logística e uma regra simples de taxa de atraso por tipo de demanda."),
    "@@CRITERIO@@": tex_esc(f"Definiu-se, antes da execução, que a hipótese é confirmada se (C1) o ROC-AUC do modelo no teste temporal for maior ou igual a {fmt(LIM_AUC, 2)} e (C2) a mediana do valor absoluto da correlação de Spearman entre a fila da agência e o tempo de resolução (uma correlação por agência) for menor que {fmt(LIM_RHO, 2)}; parcialmente confirmada se apenas um critério for atendido; refutada se nenhum for."),
    "@@P1@@": tex_esc(txt_p1()), "@@P2@@": tex_esc(txt_p2()), "@@P3@@": tex_esc(txt_p3()), "@@P4@@": tex_esc(txt_p4()), "@@P5@@": tex_esc(txt_p5()),
    "@@FIGP1@@": tex_fig("p1.png", "Chamados abertos por mês e percentual resolvido em mais de 7 dias.", "fig:p1"),
    "@@FIGP2@@": tex_fig("p2.png", "Tempo de resolução por agência e tipos de demanda que mais geram atrasos.", "fig:p2"),
    "@@FIGP3@@": tex_fig("p3.png", "Atrasos segundo dia da semana, mês e hora de abertura.", "fig:p3"),
    "@@FIGP4@@": tex_fig("p4.png", "Correlação entre fila e tempo de resolução, por agência.", "fig:p4"),
    "@@FIGP5@@": tex_fig("p5.png", "Curvas ROC no teste temporal e importância das variáveis.", "fig:p5"),
    "@@TABA@@": tex_tab(_ta, "lrrrr", "Agências com mais chamados fechados.", "tab:ag"),
    "@@TABM@@": tex_tab(_tm, "lrrr", "Desempenho no teste temporal (out–dez).", "tab:mod"),
    "@@GERAL@@": tex_esc(txt_geral()),
    "@@QUAL@@": tex_esc("A análise de qualidade cobriu completude, unicidade, conformidade, consistência, acurácia e atualidade; os resultados detalhados estão no repositório (README e figuras de qualidade)."),
    "@@LIMITES@@": "\n".join(r"\item " + tex_esc(x) for x in _lim),
    "@@CONCL@@": tex_esc(f"Nos dados de {INICIO.year}, a hipótese de que o atraso é previsível na abertura e pouco dependente da carga do dia é {VEREDITO}, segundo o critério declarado. "
                         "Como trabalhos futuros, sugerem-se a previsão do tempo de resolução (regressão ou análise de sobrevivência), a inclusão de variáveis de processo, clima e calendário, e a validação em mais de um período e em outras cidades."),
}
for _k, _v in _sub.items():
    TEX = TEX.replace(_k, _v)
assert "@@" not in TEX
with open(f"{SAIDA}/short_paper.tex", "w", encoding="utf-8") as f:
    f.write(TEX)

In [ ]:
# ═══ CÉLULA 15c · LICENSE ═══
LICENSE = f"""MIT License

Copyright (c) {datetime.date.today().year} Daniel de Carvalho Vianna

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

---
Licença dos dados (não redistribuídos neste repositório):
- NYC Open Data · 311 Service Requests: dados abertos da cidade de Nova York; ver {LICENCA_URL}
"""
with open(f"{SAIDA}/LICENSE", "w", encoding="utf-8") as f:
    f.write(LICENSE)

## Célula 16 · Conferência final, `.zip` e download
Primeiro baixa-se o **`saida.zip`** (um único arquivo, com tudo). Em seguida o notebook tenta baixar cada arquivo solto; se o navegador bloquear downloads múltiplos, use o zip. Os gráficos também aparecem nas saídas das células acima.

In [ ]:
# ═══ CÉLULA 16 · Zip e download ═══
ESPERADOS = ["README.md", "catalogo_dados.md", "short_paper.tex", "LICENSE", "coleta_meta.json", "persistencia.png",
             "qualidade_1_completude.png", "qualidade_2_unicidade_conformidade_consistencia.png", "qualidade_3_acuracia_atualidade.png",
             "p1.png", "p2.png", "p3.png", "p4.png", "p5.png"]
faltam = [a for a in ESPERADOS if not os.path.exists(f"{SAIDA}/{a}")]
assert not faltam, f"arquivos não gerados: {faltam}"
pend = [l for l in open(f"{SAIDA}/README.md", encoding="utf-8") if "PREENCHER" in l or "@@" in l]
assert not pend, "README contém campos não preenchidos"
ZIP = shutil.make_archive(f"{WORK}/saida", "zip", SAIDA)
print("Arquivos em saida/:", sorted(os.listdir(SAIDA)))
print("ZIP:", ZIP)
if not MODO_TESTE:
    from google.colab import files
    files.download(ZIP)
    time.sleep(3)
    for a in ESPERADOS:
        files.download(f"{SAIDA}/{a}")
        time.sleep(1)